# DetBayesRTMMRL vs BayesAdapter: MSP uncertainty, R/C feature analysis, and text-image similarity, R/C/MixFeat analysis, and prototype-level OOD evidence chain

本 notebook 是对原 BayesRTMMRL/BayesAdapter 分析脚本 的结构性改版，核心改动如下：

1. **主不确定性分数固定为 MSP uncertainty**：`1 - max softmax probability`。
2. **不再画 confidence histogram**，所有分布图使用 KDE/ECDF 曲线。
3. **每个 seed 单独作为一个 subplot**，不做 seed 聚合主图。
4. **DetBayesRTMMRL 的 R 分支和 C 分支都进入特征分析**：embedding、similarity heatmap、similarity distribution 都分别绘制。
5. **DetBayesRTMMRL 与 BayesAdapter 必须同图对比**，避免分开画导致不可比。
6. 输出包括 OOD detection、calibration、ID error detection、risk-coverage、feature similarity statistics。

本 notebook 只做 evaluation / analysis，不训练模型。

**v7 图形版说明**：本 notebook 会在运行时记录 DetBayesRTMMRL 与 BayesAdapter 实际构建出的 method/model 类、源码文件、`forward_eval`、`forward_train`、`model.forward_joint` 等关键函数位置与源码摘要，避免只通过方法名替换造成伪对比。


**v7 图形版说明**：所有多面板图统一改为“大尺寸方形子图”：每个 subplot 使用 `set_box_aspect(1)` 固定为正方形，整体 `figsize` 随行列数放大；heatmap、embedding、distribution、calibration、risk-coverage 图均按此规则输出，避免子图过窄、过挤。


**v8 跨模态矩阵版说明**：新增真正的 `text/class prototype × image feature prototype` 矩阵。DetBayesRTMMRL 的 R/C/MixFeat 分支分别与 Det 的 `text_features` 计算相似度；BayesAdapter 使用其 posterior mean class prototypes。原有 prototype/group feature 相似度图仍保留为补充，不再作为“文本-图像相似度矩阵”解释。


## 本版修改说明（v6 audited）

本版重新审查并修正了 DetBayesRTMMRL 的 `MixFeat` 与汇总逻辑。重点修正如下：

1. `Det-MixFeat` 只按清晰的三步构造，不再使用容易误解的嵌套表达：

```python
# C、R 分别表示 DetBayesRTMMRL 的 main/C 分支图像特征和 rep/R 分支图像特征。
# 当前仓库 forward_joint 返回的 C/R 特征已经是 L2-normalized；这里的 unit() 是安全检查。
C_unit = unit(C)
R_unit = unit(R)
blend = alpha * C_unit + (1 - alpha) * R_unit
MixFeat = unit(blend)
```

2. `MixFeat` 构造改为严格对齐：C/R 的样本数和维度必须一致；若不一致，直接给出 warning 并跳过该 batch，避免静默截断导致错位融合。
3. OOD MSP 汇总表修正了 `status` 列缺失时会报错的问题，并保留原本“同一 score_name 下比较 DetBayesRTMMRL 与 BayesAdapter”的口径。
4. similarity 统计修正了 group/dataset 指标命名不一致导致的 NaN 问题：同时保留明确的 `*_ood_group_*` 指标和向后兼容的旧列名。
5. similarity delta 修正了 `MixFeat` 分支无法匹配的问题，`R/C/MixFeat` 都会与 BayesAdapter adapter 分支比较。
6. embedding 和 similarity 相关图、表的文件名与索引均包含 `MixFeat`，避免输出文件名与内容不一致。

本 notebook 仍只做静态语法和逻辑口径检查；完整数值结果需要在包含本地仓库、checkpoint 和数据集的环境中运行。



In [1]:

# =========================
# 0. 配置区
# =========================
from pathlib import Path

NOTEBOOK_VERSION = "original_v8_logic_separate_vector_2026_06_21"

REPO_ROOT = Path("/root/autodl-tmp/MMRL").expanduser().resolve()
DATASET_ROOT = REPO_ROOT / "DATASETS"
OUTPUT_ROOT = REPO_ROOT / "output_refactor"

ID_DATASETS = ["cifar_10"]
OOD_DATASETS = ["dtd"]

PROTOCOL = "FS"
SHOTS = [ 16]
SEEDS = [1]
BACKBONE = "ViT-B/16"

# 默认不允许把 DetBayesRTMMRL 静默回退到 BayesRTMMRL 配置；
# 如果你的 Det 实现确实复用同一个 yaml，手动改成 True。
ALLOW_DET_CONFIG_FALLBACK = False

# DetBayesRTMMRL -> online, BayesAdapter -> cache，与原脚本保持一致。
EXEC_MODE_BY_METHOD = {
    "DetBayesRTMMRL": "online",
    "BayesAdapter": "cache",
}

LOAD_EPOCH = None
OOD_BATCH_SIZE = 250
OOD_NUM_WORKERS = 4

# 缓存控制
RECOMPUTE_CACHE = True       # True: 即使已有缓存也重新 forward
SAVE_FEATURES = True         # 必须 True，否则无法画 R/C 分支和 similarity matrix
KEEP_ON_CPU = True

# 主分数：固定为 MSP uncertainty
MAIN_UNCERTAINTY_SCORE = "msp_uncertainty"
HISTOGRAM_ENABLED = False
DISTRIBUTION_PLOT_TYPE = "KDE"   # KDE 或 ECDF；主图默认 KDE，必要时可补 ECDF

# DetBayesRTMMRL 特征分支。会优先从 outputs.features 中找这些 key 的候选名。
FEATURE_BRANCHES = {
    # Det-MixFeat is constructed in collect_selected_outputs():
    #   C_unit = unit(C); R_unit = unit(R)
    #   blend = alpha * C_unit + (1-alpha) * R_unit
    #   MixFeat = unit(blend)
    "DetBayesRTMMRL": ["R", "C", "MixFeat"],
    "BayesAdapter": ["adapter"],
}

# 不同项目实现中，features 字典的 key 可能不同。按顺序匹配。
FEATURE_KEY_CANDIDATES = {
    "R": ["R", "r", "rep", "feat_R", "features_R", "image_features_R", "image_features_rep", "r_features", "robust", "robust_features", "z_R", "r_branch"],
    "C": ["C", "c", "main", "img", "image", "feat_C", "features_C", "image_features_C", "image_features_main", "c_features", "class", "class_features", "z_C", "c_branch"],
    "adapter": ["adapter", "img", "image", "image_features", "features", "feat", "z"],
    "MixFeat": ["MixFeat", "mixfeat", "mixed", "mixed_feature", "mixed_features", "f_mix", "image_features_mix"],
    "fused": ["fused", "fusion", "img", "image_features_main", "image_features", "features", "feat", "z"],
}

# 可视化采样，避免 UMAP/t-SNE 过大。
MAX_ID_PER_CLASS_FOR_FEATURE = 80
MAX_OOD_FOR_FEATURE = 800

# Prototype-level similarity matrix 采样。
# 不再展示 sample-sample 全矩阵；ID 每个 class 取 20 个样本，
# OOD 每个可用 class 取 20 个样本。ID-OOD 使用一个 OOD dataset prototype，
# 避免生成 10 x OOD类别数的不可比矩阵。
PROTOTYPE_ID_PER_CLASS = 20
PROTOTYPE_OOD_PER_CLASS = 20

# Similarity heatmap 修正版配置。
# 当 OOD loader 不提供真实类别标签或所有标签都相同，不能再画 1 x 1 的
# “OOD class-class”；此时用 OOD feature 聚类/分组 prototype 代替。
PROTOTYPE_OOD_GROUPS = 10
PROTOTYPE_OOD_PER_GROUP = PROTOTYPE_OOD_PER_CLASS
PROTOTYPE_OOD_MAX_TOTAL_FOR_GROUPING = 1000
SIM_HEATMAP_AUTO_SCALE = True
SIM_HEATMAP_PERCENTILES = (2, 98)
SIM_HEATMAP_EXCLUDE_DIAGONAL_FOR_SCALE = True

# 旧变量保留作兼容，不再用于 similarity heatmap。
SIM_MATRIX_ID_PER_CLASS = PROTOTYPE_ID_PER_CLASS
SIM_MATRIX_OOD_TOTAL = 500
RANDOM_STATE = 2026

# 大图与方形子图配置。所有多面板图均按“每个 subplot 近似正方形”生成。
FIG_PANEL_SIZE = 6.2              # distribution / embedding / calibration / risk-coverage 每个子图边长，单位 inch
FIG_HEATMAP_PANEL_SIZE = 5.6      # similarity heatmap 每个子图边长，单位 inch
FIG_COLORBAR_EXTRA_WIDTH = 1.2    # heatmap 色条额外宽度
FIG_TOP_EXTRA_HEIGHT = 0.9        # suptitle 额外高度
FIG_DPI = 300

# 输出目录
ANALYSIS_ROOT = OUTPUT_ROOT / "analysis" / "original_v8_logic_separate_vector"
CACHE_ROOT = ANALYSIS_ROOT / "caches"
PRED_CACHE_ROOT = CACHE_ROOT / "predictions"
FEATURE_CACHE_ROOT = CACHE_ROOT / "features"
UNCERTAINTY_CACHE_ROOT = CACHE_ROOT / "uncertainty"
SUMMARY_ROOT = ANALYSIS_ROOT / "summaries"
FIGURE_ROOT = ANALYSIS_ROOT / "figures"
PAPER_READY_ROOT = ANALYSIS_ROOT / "paper_ready"

FIG_DIRS = {
    "msp_kde": FIGURE_ROOT / "msp_uncertainty_kde",
    "msp_ecdf": FIGURE_ROOT / "msp_uncertainty_ecdf",
    "msp_cwo": FIGURE_ROOT / "msp_correct_wrong_ood",
    "feature_embedding": FIGURE_ROOT / "feature_embedding_R_C_MixFeat_adapter",
    "similarity_matrix": FIGURE_ROOT / "similarity_matrix_R_C_MixFeat_adapter",
    "text_image_matrix": FIGURE_ROOT / "text_image_matrix_R_C_MixFeat_adapter",
    "similarity_distribution": FIGURE_ROOT / "similarity_distribution_R_C_MixFeat_adapter",
    "calibration": FIGURE_ROOT / "calibration",
    "risk_coverage": FIGURE_ROOT / "risk_coverage",
    "metric_delta": FIGURE_ROOT / "metric_delta_msp",
}

for p in [CACHE_ROOT, PRED_CACHE_ROOT, FEATURE_CACHE_ROOT, UNCERTAINTY_CACHE_ROOT, SUMMARY_ROOT, FIGURE_ROOT, PAPER_READY_ROOT, *FIG_DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_VERSION:", NOTEBOOK_VERSION)
print("REPO_ROOT:", REPO_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("ANALYSIS_ROOT:", ANALYSIS_ROOT)
print("MAIN_UNCERTAINTY_SCORE:", MAIN_UNCERTAINTY_SCORE)


NOTEBOOK_VERSION: original_v8_logic_separate_vector_2026_06_21
REPO_ROOT: /root/autodl-tmp/MMRL
DATASET_ROOT: /root/autodl-tmp/MMRL/DATASETS
OUTPUT_ROOT: /root/autodl-tmp/MMRL/output_refactor
ANALYSIS_ROOT: /root/autodl-tmp/MMRL/output_refactor/analysis/original_v8_logic_separate_vector
MAIN_UNCERTAINTY_SCORE: msp_uncertainty


## 1. 环境初始化

这一步只加载项目模块和常用分析库。不会训练模型。



In [2]:

# =========================
# 1. 环境初始化
# =========================
import os
import sys
import json
import math
import importlib
import inspect
import hashlib
import warnings
from argparse import Namespace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("TORCH_NUM_THREADS", "1")
os.environ.setdefault("TORCH_NUM_INTEROP_THREADS", "1")

REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dassl.engine import build_trainer
from dassl.utils import set_random_seed, setup_logger

from core.config import setup_cfg
from core.utils import import_optional_modules
from eval_ood import build_ood_loader

# sklearn 用于 AUROC/AUPR/calibration/risk-coverage。
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
)

try:
    from scipy.stats import gaussian_kde
    SCIPY_AVAILABLE = True
except Exception:
    gaussian_kde = None
    SCIPY_AVAILABLE = False

try:
    from sklearn.manifold import TSNE
    from sklearn.decomposition import PCA
    SKLEARN_EMBED_AVAILABLE = True
except Exception:
    TSNE = None
    PCA = None
    SKLEARN_EMBED_AVAILABLE = False

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    umap = None
    UMAP_AVAILABLE = False


def import_runtime_modules():
    import_optional_modules([
        "datasets.cifar_10",
        "datasets.ood_image_datasets",
        "datasets.oxford_pets",
        "datasets.oxford_flowers",
        "datasets.fgvc_aircraft",
        "datasets.dtd",
        "datasets.eurosat",
        "datasets.stanford_cars",
        "datasets.food101",
        "datasets.sun397",
        "datasets.caltech101",
        "datasets.ucf101",
        "datasets.imagenet",
        "datasets.imagenetv2",
        "datasets.imagenet_sketch",
        "datasets.imagenet_a",
        "datasets.imagenet_r",
    ])
    importlib.import_module("trainers.refactor_runner")

import_runtime_modules()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

try:
    import subprocess
    git_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT).decode().strip()
except Exception as e:
    git_head = f"unknown: {e}"
print("git_head =", git_head)
print("SCIPY_AVAILABLE =", SCIPY_AVAILABLE)
print("UMAP_AVAILABLE =", UMAP_AVAILABLE)
print("SKLEARN_EMBED_AVAILABLE =", SKLEARN_EMBED_AVAILABLE)

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)
warnings.filterwarnings("ignore", category=UserWarning)



device = cuda
git_head = 672f616e7437fb1d2261ea4cc4f1501f3633e8f1
SCIPY_AVAILABLE = True
UMAP_AVAILABLE = False
SKLEARN_EMBED_AVAILABLE = True


## 2. 方法配置、checkpoint 路径与 cache 路径

路径规则与原 notebook 保持一致：

- `DetBayesRTMMRL`：`output_refactor/DetBayesRTMMRL/FS/fewshot_train/<dataset>/shots_<shot>/ViT-B-16/default/seed<seed>`
- `BayesAdapter`：`output_refactor/ClipAdapters/BAYES_ADAPTER/FS/fewshot_train/<dataset>/shots_<shot>/ViT-B-16/seed<seed>`



In [3]:

# =========================
# 2. 方法配置与路径规则
# =========================
METHOD_SPECS = {
    "DetBayesRTMMRL": {
        "requested_method": "DetBayesRTMMRL",
        "launch_method": "DetBayesRTMMRL",
        # 优先使用 DetBayesRTMMRL 专属配置；默认不会静默回退到 BayesRTMMRL 配置。
        "method_config_file": "configs/methods/detbayesrt_mmrl.yaml",
        "method_config_file_candidates": [
            "configs/methods/detbayesrt_mmrl.yaml",
            "configs/methods/det_bayesrt_mmrl.yaml",
            "configs/methods/detbayesrtmmrl.yaml",
        ],
        "runtime_config_file": "configs/runtime/mmrl_family.yaml",
        "runtime_config_file_candidates": [
            "configs/runtime/detbayesrt_mmrl_family.yaml",
            "configs/runtime/mmrl_family.yaml",
        ],
        "run_tag": "default",
        "is_adapter": False,
    },
    "BayesAdapter": {
        "requested_method": "BayesAdapter",
        "launch_method": "ClipAdapters",
        "method_config_file": "configs/methods/clip_adapters_bayes.yaml",
        "method_config_file_candidates": [
            "configs/methods/clip_adapters_bayes.yaml",
        ],
        "runtime_config_file": "configs/runtime/adapter_family.yaml",
        "runtime_config_file_candidates": [
            "configs/runtime/adapter_family.yaml",
        ],
        "run_tag": "BAYES_ADAPTER",
        "is_adapter": True,
    },
}

# 若 DetBayesRTMMRL 在你的仓库中明确复用 bayesrt_mmrl.yaml，再手动打开配置区的开关。
if globals().get("ALLOW_DET_CONFIG_FALLBACK", False):
    METHOD_SPECS["DetBayesRTMMRL"]["method_config_file_candidates"].append("configs/methods/bayesrt_mmrl.yaml")


def resolve_repo_config(spec: dict, key: str) -> Path:
    """Return the first existing config path for key.

    key is "method_config_file" or "runtime_config_file".
    """
    candidates = spec.get(f"{key}_candidates", None) or [spec[key]]
    checked = []
    for rel in candidates:
        p = (REPO_ROOT / rel).expanduser()
        checked.append(str(p))
        if p.exists():
            if str(rel) != str(spec.get(key, rel)):
                print(f"[INFO] fallback {key}: {rel}")
            return p
    raise FileNotFoundError(
        f"Cannot resolve {key} for method={spec.get('requested_method')}. Checked:\n"
        + "\n".join(checked)
    )


def protocol_phase(protocol: str):
    protocol = str(protocol).upper()
    if protocol == "FS":
        return "fewshot_train", "all"
    if protocol == "B2N":
        return "train_base", "base"
    if protocol == "CD":
        return "cross_train", "all"
    raise ValueError(f"Unsupported protocol: {protocol}")


def dataset_config_file(dataset: str) -> Path:
    return REPO_ROOT / "configs" / "datasets" / f"{dataset}.yaml"


def protocol_config_file(protocol: str) -> Path:
    protocol = str(protocol).upper()
    if protocol == "FS":
        return REPO_ROOT / "configs" / "protocols" / "fs.yaml"
    if protocol == "B2N":
        return REPO_ROOT / "configs" / "protocols" / "b2n.yaml"
    if protocol == "CD":
        return REPO_ROOT / "configs" / "protocols" / "cd.yaml"
    raise ValueError(f"Unsupported protocol: {protocol}")


def backbone_dir(backbone: str) -> str:
    return str(backbone).replace("/", "-")


def build_model_dir(method_name: str, dataset: str, shot: int, seed: int) -> Path:
    spec = METHOD_SPECS[method_name]
    phase, _subsample = protocol_phase(PROTOCOL)
    bdir = backbone_dir(BACKBONE)
    if spec["is_adapter"]:
        return OUTPUT_ROOT / spec["launch_method"] / spec["run_tag"] / PROTOCOL / phase / dataset / f"shots_{shot}" / bdir / f"seed{seed}"
    return OUTPUT_ROOT / spec["launch_method"] / PROTOCOL / phase / dataset / f"shots_{shot}" / bdir / spec["run_tag"] / f"seed{seed}"


def build_case_cache_dir(method_name: str, id_dataset: str, shot: int, seed: int) -> Path:
    return PRED_CACHE_ROOT / method_name / PROTOCOL / id_dataset / f"shots_{shot}" / backbone_dir(BACKBONE) / f"seed{seed}"


def build_feature_cache_dir(method_name: str, feature_branch: str, id_dataset: str, shot: int, seed: int) -> Path:
    return FEATURE_CACHE_ROOT / method_name / feature_branch / PROTOCOL / id_dataset / f"shots_{shot}" / backbone_dir(BACKBONE) / f"seed{seed}"


for m in METHOD_SPECS:
    print(m, METHOD_SPECS[m])



DetBayesRTMMRL {'requested_method': 'DetBayesRTMMRL', 'launch_method': 'DetBayesRTMMRL', 'method_config_file': 'configs/methods/detbayesrt_mmrl.yaml', 'method_config_file_candidates': ['configs/methods/detbayesrt_mmrl.yaml', 'configs/methods/det_bayesrt_mmrl.yaml', 'configs/methods/detbayesrtmmrl.yaml'], 'runtime_config_file': 'configs/runtime/mmrl_family.yaml', 'runtime_config_file_candidates': ['configs/runtime/detbayesrt_mmrl_family.yaml', 'configs/runtime/mmrl_family.yaml'], 'run_tag': 'default', 'is_adapter': False}
BayesAdapter {'requested_method': 'BayesAdapter', 'launch_method': 'ClipAdapters', 'method_config_file': 'configs/methods/clip_adapters_bayes.yaml', 'method_config_file_candidates': ['configs/methods/clip_adapters_bayes.yaml'], 'runtime_config_file': 'configs/runtime/adapter_family.yaml', 'runtime_config_file_candidates': ['configs/runtime/adapter_family.yaml'], 'run_tag': 'BAYES_ADAPTER', 'is_adapter': True}


## 3. 构建 trainer 与加载 checkpoint

此部分不训练，只构建模型并加载已有 checkpoint。缺失 checkpoint 会记录到 summary，不会中断整个批处理。



In [4]:

# =========================
# 3. 构建 trainer
# =========================
def make_args(method_name: str, id_dataset: str, shot: int, seed: int, model_dir: Path, output_dir: Path):
    spec = METHOD_SPECS[method_name]
    _phase, subsample = protocol_phase(PROTOCOL)
    exec_mode = EXEC_MODE_BY_METHOD.get(method_name, "online")

    dcfg = dataset_config_file(id_dataset)
    pcfg = protocol_config_file(PROTOCOL)
    mcfg = resolve_repo_config(spec, "method_config_file")
    rcfg = resolve_repo_config(spec, "runtime_config_file")

    for path in [dcfg, pcfg, mcfg, rcfg]:
        if not path.exists():
            raise FileNotFoundError(path)

    opts = [
        "DATASET.NUM_SHOTS", str(shot),
        "DATASET.SUBSAMPLE_CLASSES", subsample,
        "MODEL.BACKBONE.NAME", BACKBONE,
    ]

    return Namespace(
        root=str(DATASET_ROOT),
        output_dir=str(output_dir),
        dataset_config_file=str(dcfg),
        method_config_file=str(mcfg),
        protocol_config_file=str(pcfg),
        runtime_config_file=str(rcfg),
        exp_config="",
        method=spec["launch_method"],
        protocol=PROTOCOL,
        exec_mode=exec_mode,
        seed=int(seed),
        trainer="RefactorRunner",
        eval_only=True,
        model_dir=str(model_dir),
        load_epoch=LOAD_EPOCH,
        no_train=True,
        opts=opts,
    )


def build_loaded_trainer(method_name: str, id_dataset: str, shot: int, seed: int):
    model_dir = build_model_dir(method_name, id_dataset, shot, seed)
    cache_dir = build_case_cache_dir(method_name, id_dataset, shot, seed)
    cache_dir.mkdir(parents=True, exist_ok=True)

    if not model_dir.exists():
        raise FileNotFoundError(f"checkpoint dir not found: {model_dir}")

    args = make_args(
        method_name=method_name,
        id_dataset=id_dataset,
        shot=shot,
        seed=seed,
        model_dir=model_dir,
        output_dir=cache_dir / "_runtime_output",
    )

    if int(seed) >= 0:
        set_random_seed(int(seed))

    cfg = setup_cfg(args)
    setup_logger(cfg.OUTPUT_DIR)

    trainer = build_trainer(cfg)
    trainer.load_model(str(model_dir), epoch=LOAD_EPOCH)
    trainer.set_model_mode("eval")
    return trainer, model_dir, cache_dir



In [5]:

# =========================
# 3b. 运行时实现检查：记录每个 method 实际类、源码位置、关键 forward/train/eval 函数
# =========================
def _callable_source_info(obj, max_source_chars=5000):
    info = {
        "exists": obj is not None,
        "qualname": "",
        "module": "",
        "file": "",
        "line": None,
        "signature": "",
        "source_sha1": "",
        "source_excerpt": "",
        "error": "",
    }
    if obj is None:
        return info
    try:
        target = getattr(obj, "__func__", obj)
        info["qualname"] = getattr(target, "__qualname__", repr(target))
        info["module"] = getattr(target, "__module__", "")
        try:
            info["signature"] = str(inspect.signature(target))
        except Exception as e:
            info["signature"] = f"<signature unavailable: {e!r}>"
        try:
            info["file"] = inspect.getsourcefile(target) or ""
            lines, line_no = inspect.getsourcelines(target)
            source_text = "".join(lines)
            info["line"] = int(line_no)
            info["source_sha1"] = hashlib.sha1(source_text.encode("utf-8", errors="ignore")).hexdigest()
            info["source_excerpt"] = source_text[:max_source_chars]
        except Exception as e:
            info["error"] = f"source unavailable: {e!r}"
    except Exception as e:
        info["error"] = repr(e)
    return info


def runtime_implementation_report(trainer, method_name, id_dataset, shot, seed, model_dir):
    method = getattr(trainer, "method", None)
    model = getattr(method, "model", None)
    report = {
        "method_requested": method_name,
        "id_dataset": id_dataset,
        "shot": int(shot),
        "seed": int(seed),
        "model_dir": str(model_dir),
        "trainer_class": type(trainer).__module__ + "." + type(trainer).__qualname__,
        "method_class": "" if method is None else type(method).__module__ + "." + type(method).__qualname__,
        "model_class": "" if model is None else type(model).__module__ + "." + type(model).__qualname__,
        "cfg_method_name": str(getattr(getattr(trainer, "cfg", None), "METHOD", "")),
        "device": str(getattr(trainer, "device", "")),
        "checkpoints_loaded_from": str(model_dir),
        "test_loader_class": type(getattr(trainer, "test_loader", None)).__name__,
        "functions": {
            "method.forward_eval": _callable_source_info(getattr(method, "forward_eval", None)),
            "method.select_eval_logits": _callable_source_info(getattr(method, "select_eval_logits", None)),
            "method.forward_train": _callable_source_info(getattr(method, "forward_train", None)),
            "model.forward": _callable_source_info(getattr(model, "forward", None)),
            "model.forward_joint": _callable_source_info(getattr(model, "forward_joint", None)),
        },
    }
    return report


def save_runtime_implementation_report(report, cache_dir):
    path = Path(cache_dir) / "runtime_implementation_report.json"
    with path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    flat_rows = []
    for name, info in report.get("functions", {}).items():
        row = {k: v for k, v in report.items() if k != "functions"}
        row.update({"function": name, **{k: v for k, v in info.items() if k != "source_excerpt"}})
        flat_rows.append(row)
    pd.DataFrame(flat_rows).to_csv(Path(cache_dir) / "runtime_implementation_report.csv", index=False)
    print("[IMPL]", report["method_requested"], "method_class=", report["method_class"], "model_class=", report["model_class"])
    for name, info in report.get("functions", {}).items():
        print(f"[IMPL] {name}: {info.get('file','')}:{info.get('line','')} {info.get('signature','')}")
    return path



## 4. MSP uncertainty 与基础指标函数

主分数统一为：

```text
MSP confidence = max_c softmax(logits)_c
MSP uncertainty = 1 - MSP confidence
```

OOD detection 中分数方向固定为：

```text
score 越大 = 越不确定 = 越倾向 OOD / 错误样本
```



In [6]:

# =========================
# 4. MSP uncertainty 与指标函数
# =========================
def softmax_numpy(logits):
    x = np.asarray(logits, dtype=np.float64)
    x = x - np.max(x, axis=1, keepdims=True)
    e = np.exp(x)
    return e / np.clip(e.sum(axis=1, keepdims=True), 1e-12, None)


def probs_from_logits_tensor(logits_t: torch.Tensor) -> torch.Tensor:
    return F.softmax(logits_t.float(), dim=1)


def msp_confidence_from_logits(logits_t: torch.Tensor) -> torch.Tensor:
    return probs_from_logits_tensor(logits_t).max(dim=1).values.detach().float().cpu()


def msp_uncertainty_from_logits(logits_t: torch.Tensor) -> torch.Tensor:
    return (1.0 - msp_confidence_from_logits(logits_t)).detach().float().cpu()


def predictive_entropy_from_logits(logits_t: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    p = probs_from_logits_tensor(logits_t).clamp_min(eps)
    return (-(p * p.log()).sum(dim=1)).detach().float().cpu()


def top1_pred_from_logits(logits_t: torch.Tensor) -> torch.Tensor:
    return logits_t.float().argmax(dim=1).detach().long().cpu()


def fpr_at_tpr(y_true, scores, target_tpr=0.95):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)
    order = np.argsort(-scores)
    y = y_true[order]
    pos = max(int((y == 1).sum()), 1)
    neg = max(int((y == 0).sum()), 1)
    tp = np.cumsum(y == 1)
    fp = np.cumsum(y == 0)
    tpr = tp / pos
    fpr = fp / neg
    idx = np.where(tpr >= target_tpr)[0]
    if len(idx) == 0:
        return np.nan
    return float(np.min(fpr[idx]))


def detection_error(y_true, scores):
    """Minimum detection error with equal class prior; positive class has larger score."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)
    thresholds = np.r_[np.inf, np.sort(np.unique(scores))[::-1], -np.inf]
    pos = max(int((y_true == 1).sum()), 1)
    neg = max(int((y_true == 0).sum()), 1)
    best = np.inf
    for t in thresholds:
        pred_pos = scores >= t
        fnr = ((~pred_pos) & (y_true == 1)).sum() / pos
        fpr = (pred_pos & (y_true == 0)).sum() / neg
        best = min(best, 0.5 * (fnr + fpr))
    return float(best)


def compute_ood_metrics_from_uncertainty(id_uncertainty, ood_uncertainty):
    """ID=0, OOD=1, larger score means more likely OOD."""
    id_s = np.asarray(id_uncertainty, dtype=float)
    ood_s = np.asarray(ood_uncertainty, dtype=float)
    y_true = np.r_[np.zeros_like(id_s, dtype=int), np.ones_like(ood_s, dtype=int)]
    scores = np.r_[id_s, ood_s]

    if len(np.unique(y_true)) < 2 or len(np.unique(scores)) < 2:
        auroc = np.nan
    else:
        auroc = float(roc_auc_score(y_true, scores))

    return {
        "AUROC": auroc,
        "AUPR_OUT": float(average_precision_score(y_true, scores)),
        "AUPR_IN": float(average_precision_score(1 - y_true, -scores)),
        "FPR95": fpr_at_tpr(y_true, scores, target_tpr=0.95),
        "DetectionError": detection_error(y_true, scores),
    }


def compute_error_detection_metrics(correct_bool, uncertainty):
    """ID 内部错误检测：positive=incorrect, score=uncertainty."""
    correct_bool = np.asarray(correct_bool).astype(bool)
    scores = np.asarray(uncertainty, dtype=float)
    y_true = (~correct_bool).astype(int)
    if len(np.unique(y_true)) < 2 or len(np.unique(scores)) < 2:
        return {"ErrorAUROC": np.nan, "ErrorAUPR": np.nan, "ErrorFPR95": np.nan}
    return {
        "ErrorAUROC": float(roc_auc_score(y_true, scores)),
        "ErrorAUPR": float(average_precision_score(y_true, scores)),
        "ErrorFPR95": fpr_at_tpr(y_true, scores, target_tpr=0.95),
    }



## 5. R/C 分支特征抽取工具

`DetBayesRTMMRL` 必须保存 R 分支与 C 分支特征。由于项目实现中 `outputs.features` 的 key 可能不同，这里用候选 key 列表做鲁棒匹配。

如果运行时某个分支没有匹配到特征，notebook 会在 `feature_key_report.csv` 里记录，方便你根据真实 key 修改 `FEATURE_KEY_CANDIDATES`。



In [7]:

# =========================
# 5. R/C 分支特征抽取工具
# =========================
def as_cpu_float_tensor(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().float().cpu()
    try:
        return torch.as_tensor(x).detach().float().cpu()
    except Exception:
        return None


def get_attr_or_key(obj, key, default=None):
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def extract_feature_dict_from_outputs(outputs):
    """Return a dictionary-like feature container from method outputs."""
    candidates = []
    for name in ["features", "feature", "feats", "embeddings", "embedding"]:
        val = get_attr_or_key(outputs, name, None)
        if val is not None:
            candidates.append(val)
    # Some implementations return a dict directly.
    if isinstance(outputs, dict):
        candidates.append(outputs)

    for val in candidates:
        if isinstance(val, dict):
            return val
    return {}


def pick_feature_from_dict(feature_dict, branch_name):
    candidates = FEATURE_KEY_CANDIDATES.get(branch_name, [branch_name])
    available = list(feature_dict.keys()) if isinstance(feature_dict, dict) else []

    # Exact candidates first. Exact one-letter keys such as "R"/"C" are allowed.
    for k in candidates:
        if isinstance(feature_dict, dict) and k in feature_dict:
            return as_cpu_float_tensor(feature_dict[k]), k, available

    # Case-insensitive exact match.
    lowered = {str(k).lower(): k for k in available}
    for k in candidates:
        lk = str(k).lower()
        if lk in lowered:
            real_k = lowered[lk]
            return as_cpu_float_tensor(feature_dict[real_k]), real_k, available

    # Safe substring fallback. Do NOT use one-letter candidates ("r"/"c")
    # as substrings, because they match unrelated keys and can make R/C read
    # the same tensor. Only semantically specific candidates are eligible.
    safe_candidates = [str(c).lower() for c in candidates if len(str(c)) >= 3]
    for real_k in available:
        rk = str(real_k).lower()
        for cand in safe_candidates:
            if cand in rk:
                return as_cpu_float_tensor(feature_dict[real_k]), real_k, available

    return None, None, available


def expected_branches_for_method(method_name):
    return FEATURE_BRANCHES.get(method_name, ["adapter"])


def normalize_feature_tensor_shape(feat):
    """Return a feature matrix [N, D].

    Handles the common Bayesian output shape [S, N, D] by averaging the MC/sample
    dimension. Avoids silently flattening raw images [N, 3, H, W] as features.
    """
    if feat is None:
        return None
    if not isinstance(feat, torch.Tensor):
        feat = as_cpu_float_tensor(feat)
    if feat is None:
        return None

    # Reject raw image-like tensors. Feature maps with many channels can still be pooled/flattened.
    if feat.ndim >= 4 and int(feat.shape[1]) in (1, 3) and int(feat.shape[-1]) >= 16 and int(feat.shape[-2]) >= 16:
        return None

    if feat.ndim == 1:
        return feat[:, None]

    # Typical stochastic feature tensor: [num_samples, batch, dim].
    if feat.ndim == 3 and int(feat.shape[0]) <= 64 and int(feat.shape[1]) >= 1:
        return feat.float().mean(dim=0)

    # Less common layout: [batch, num_samples, dim].
    if feat.ndim == 3 and int(feat.shape[1]) <= 64 and int(feat.shape[0]) > int(feat.shape[1]):
        return feat.float().mean(dim=1)

    if feat.ndim > 2:
        return feat.flatten(start_dim=1)
    return feat


def l2_normalize_np(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float64)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(n, eps, None)


def l2_normalize_tensor(x, eps=1e-12):
    """L2-normalize a feature matrix [N, D] on the last dimension."""
    if x is None:
        return None
    x = as_cpu_float_tensor(x)
    if x is None:
        return None
    return F.normalize(x.float(), dim=-1, eps=float(eps))


def get_bayesrt_alpha(method, default=0.5):
    """Read BAYESRT_MMRL.ALPHA for C/R feature fusion."""
    try:
        cfg = get_bayes_family_cfg(method)
        if cfg is not None and hasattr(cfg, "ALPHA"):
            return float(getattr(cfg, "ALPHA"))
    except Exception:
        pass
    return float(default)


def build_det_mixfeat_tensor(c_feat, r_feat, alpha):
    """Construct Det-MixFeat as an aligned unit-vector interpolation of C/R features.

    Semantics:
        C_unit = unit(C)
        R_unit = unit(R)
        blend = alpha * C_unit + (1-alpha) * R_unit
        MixFeat = unit(blend)

    Important implementation guard:
        C and R must describe the same samples in the same order. Therefore the
        first dimension and feature dimension must match exactly. We intentionally
        do not truncate to the shorter tensor, because silent truncation can hide
        feature/label or batch-alignment bugs.
    """
    c_feat = normalize_feature_tensor_shape(c_feat)
    r_feat = normalize_feature_tensor_shape(r_feat)
    if c_feat is None or r_feat is None:
        return None

    if c_feat.ndim != 2 or r_feat.ndim != 2:
        print(
            f"[WARN] Cannot build Det-MixFeat because C/R are not 2-D feature matrices: "
            f"C={tuple(c_feat.shape)}, R={tuple(r_feat.shape)}"
        )
        return None

    if int(c_feat.shape[0]) != int(r_feat.shape[0]) or int(c_feat.shape[1]) != int(r_feat.shape[1]):
        print(
            f"[WARN] Cannot build Det-MixFeat because C/R shapes are not aligned: "
            f"C={tuple(c_feat.shape)}, R={tuple(r_feat.shape)}"
        )
        return None

    try:
        alpha = float(alpha)
    except Exception:
        print(f"[WARN] Invalid alpha for Det-MixFeat: {alpha!r}; using alpha=0.5")
        alpha = 0.5
    if not np.isfinite(alpha):
        print(f"[WARN] Non-finite alpha for Det-MixFeat: {alpha!r}; using alpha=0.5")
        alpha = 0.5

    c_unit = l2_normalize_tensor(c_feat)
    r_unit = l2_normalize_tensor(r_feat)
    blend = alpha * c_unit + (1.0 - alpha) * r_unit
    mix_feat = F.normalize(blend.float(), dim=-1, eps=1.0e-12)
    return mix_feat.detach().cpu()



TEXT_PROTOTYPE_KEY_CANDIDATES = [
    "text_features",
    "text_feature",
    "text",
    "class_prototypes",
    "prototypes",
    "text_prototypes",
    "zeroshot_weights",
    "zs_weights",
]


def pick_tensor_by_candidates(container, candidates):
    """Pick a tensor-like object by exact/case-insensitive key from a dict or object."""
    if container is None:
        return None, None, []

    if isinstance(container, dict):
        available = list(container.keys())
        for k in candidates:
            if k in container:
                return as_cpu_float_tensor(container[k]), str(k), [str(x) for x in available]
        lowered = {str(k).lower(): k for k in available}
        for k in candidates:
            lk = str(k).lower()
            if lk in lowered:
                real_k = lowered[lk]
                return as_cpu_float_tensor(container[real_k]), str(real_k), [str(x) for x in available]
        return None, None, [str(x) for x in available]

    available = []
    for k in candidates:
        if hasattr(container, k):
            available.append(k)
            return as_cpu_float_tensor(getattr(container, k)), str(k), available
    return None, None, available


def normalize_class_prototype_tensor(x):
    """Return class/text prototypes as a 2-D CPU float tensor [num_classes, dim]."""
    x = normalize_feature_tensor_shape(x)
    if x is None:
        return None
    if not isinstance(x, torch.Tensor):
        x = as_cpu_float_tensor(x)
    if x is None or x.ndim != 2:
        return None
    return x.detach().float().cpu()


def extract_runtime_class_prototypes(method, method_name=None, outputs=None):
    """Extract text/class prototypes used for cross-modal image-text matrices.

    DetBayesRTMMRL:
        Prefer MethodOutputs/features text entries if available; the direct
        forward_joint path is handled separately in extract_bayesrt_rc_features.

    BayesAdapter:
        Prefer adapter.get_prototypes(), which returns the posterior mean class
        prototypes [C, D]. This is the deterministic class/text side used for
        the BayesAdapter text-image similarity matrix.
    """
    method_name = str(method_name or "")

    # First try outputs.features-like containers.
    feat_dict = extract_feature_dict_from_outputs(outputs)
    proto, key, _available = pick_tensor_by_candidates(feat_dict, TEXT_PROTOTYPE_KEY_CANDIDATES)
    proto = normalize_class_prototype_tensor(proto)
    if proto is not None:
        return proto, f"outputs.features.{key}"

    model = getattr(method, "model", None)
    adapter = getattr(model, "adapter", None) if model is not None else None

    if method_name == "BayesAdapter" or adapter is not None:
        if adapter is not None and hasattr(adapter, "get_prototypes"):
            try:
                proto = normalize_class_prototype_tensor(adapter.get_prototypes())
                if proto is not None:
                    return proto, "model.adapter.get_prototypes()"
            except Exception as e:
                print(f"[WARN] adapter.get_prototypes() failed: {repr(e)}")

        # Fallbacks for BayesAdapter-like implementations.
        for obj_name, obj in [("adapter", adapter), ("model", model)]:
            proto, key, _available = pick_tensor_by_candidates(obj, [
                "text_features_unnorm_mean",
                "text_features",
                "base_text_features",
                "prototypes",
                "prior_mean",
            ])
            proto = normalize_class_prototype_tensor(proto)
            if proto is not None:
                return proto, f"model.{obj_name}.{key}"

    if model is not None:
        proto, key, _available = pick_tensor_by_candidates(model, TEXT_PROTOTYPE_KEY_CANDIDATES)
        proto = normalize_class_prototype_tensor(proto)
        if proto is not None:
            return proto, f"model.{key}"

    return None, "missing_class_prototypes"




def get_bayes_family_cfg(method):
    """Return DetBayesRTMMRL/BayesRTMMRL-style config node if present."""
    cfg_root = getattr(method, "cfg", None)
    if cfg_root is None:
        return None
    for name in ["DETBAYESRT_MMRL", "BAYESRT_MMRL"]:
        cfg = getattr(cfg_root, name, None)
        if cfg is not None:
            return cfg
    return None


def bayesrt_eval_use_posterior_mean(method, eval_ctx):
    """Mirror DetBayesRTMMRLMethod.forward_eval posterior-mean policy.

    The current project exposes C branch in MethodOutputs.features["img"], but
    it does not expose the R branch there. R/C image features are available in
    DetBayesRTMMRLModel.forward_joint as:
        - image_features_rep  -> R branch
        - image_features_main -> C branch
    This helper keeps feature extraction aligned with the method's eval policy.
    """
    use_posterior_mean = bool(getattr(method, "eval_use_posterior_mean", False))
    try:
        cfg = get_bayes_family_cfg(method)
        if cfg is None:
            return use_posterior_mean
        if bool(getattr(cfg, "NOVEL_TEXT_MEAN_ONLY", False)):
            protocol = str(eval_ctx.protocol).upper()
            dataset = str(eval_ctx.dataset_name)
            sub_cls = str(eval_ctx.subsample_classes or "all")
            is_b2n_novel = protocol == "B2N" and sub_cls != "base"
            is_cd_target = protocol == "CD" and dataset != "ImageNet"
            if is_b2n_novel or is_cd_target:
                use_posterior_mean = True
    except Exception:
        pass
    return use_posterior_mean


@torch.no_grad()
def extract_bayesrt_rc_features(method, batch, eval_ctx):
    """Extract R/C features using the actual model.forward_joint implementation.

    This function does not assume BayesRTMMRL and DetBayesRTMMRL have identical
    output keys. It calls the method's real model.forward_joint and then matches
    branch keys through FEATURE_KEY_CANDIDATES. The matched key is written to the
    feature report, so a wrong implementation is visible instead of hidden.
    """
    if not hasattr(method, "model") or not hasattr(method.model, "forward_joint"):
        return {}, [], "missing_forward_joint", {}, None, None
    if not isinstance(batch, dict) or "img" not in batch:
        return {}, [], "missing_img_batch", {}, None, None

    image = batch["img"].to(method.device)
    num_samples = int(max(1, getattr(method, "n_mc_test", 1)))
    use_posterior_mean = bayesrt_eval_use_posterior_mean(method, eval_ctx)

    out = method.model.forward_joint(
        image=image,
        num_samples=num_samples,
        use_posterior_mean=use_posterior_mean,
    )

    if isinstance(out, dict):
        out_dict = out
    else:
        # Some implementations return a dataclass/namespace-like object.
        out_dict = {
            k: getattr(out, k)
            for k in dir(out)
            if not k.startswith("_") and not callable(getattr(out, k, None))
        }

    available = sorted([str(k) for k in out_dict.keys()])
    feats = {}
    matched = {}
    for branch in ["R", "C"]:
        feat, key, _available = pick_feature_from_dict(out_dict, branch)
        feat = normalize_feature_tensor_shape(feat)
        if feat is not None:
            feats[branch] = feat
            matched[branch] = f"forward_joint.{key}"

    text_proto, text_key, _text_available = pick_tensor_by_candidates(out_dict, TEXT_PROTOTYPE_KEY_CANDIDATES)
    text_proto = normalize_class_prototype_tensor(text_proto)
    text_source = f"forward_joint.{text_key}" if text_proto is not None else None
    return feats, available, "forward_joint_actual", matched, text_proto, text_source

def add_feature_chunk(features_by_branch, feature_key_report, branch, feat, matched_key, available_keys):
    feat = normalize_feature_tensor_shape(feat)
    if feat is not None:
        features_by_branch[branch].append(feat)
    if matched_key is not None:
        feature_key_report[branch]["matched_keys"].append(str(matched_key))
    feature_key_report[branch]["available_keys_examples"].append([str(x) for x in available_keys])



## 6. Forward 收集：logits、MSP uncertainty、labels、correct、features

保存 payload 的核心字段：

```python
{
    "logits": Tensor[N, C],
    "labels": Tensor[N] or None,
    "preds": Tensor[N],
    "msp_confidence": Tensor[N],
    "msp_uncertainty": Tensor[N],
    "predictive_entropy": Tensor[N],
    "correct": Tensor[N] or None,
    "features": {"R": Tensor[N,D], "C": Tensor[N,D], ...},
    "feature_key_report": {...}
}
```



In [8]:

# =========================
# 6. collector
# =========================
@torch.no_grad()
def collect_selected_outputs(trainer, loader, split_name: str, method_name: str, save_features: bool = True):
    eval_ctx = trainer.executor.build_eval_context(trainer, split_name)
    method = trainer.method
    method.eval()
    trainer.set_model_mode("eval")

    logits_all = []
    labels_all = []
    branches = expected_branches_for_method(method_name)
    features_by_branch = {b: [] for b in branches}
    feature_key_report = {b: {"matched_keys": [], "available_keys_examples": []} for b in branches}
    class_prototypes = None
    class_prototype_source = ""

    use_amp = False
    try:
        prec = method.get_precision()
        use_amp = (str(prec).lower() == "amp" and torch.cuda.is_available())
    except Exception:
        use_amp = torch.cuda.is_available()

    for batch_idx, batch in enumerate(loader):
        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = method.forward_eval(batch, eval_ctx)
            logits = method.select_eval_logits(outputs, eval_ctx)

        logits_all.append(logits.detach().float().cpu())

        labels = getattr(outputs, "labels", None)
        if labels is None and isinstance(outputs, dict):
            labels = outputs.get("labels", None)
        if labels is None and isinstance(batch, dict) and "label" in batch:
            labels = batch["label"]
        if labels is not None:
            labels_all.append(labels.detach().long().cpu())

        if save_features:
            # Repository-specific DetBayesRTMMRL path:
            # forward_eval exposes only C/main feature as outputs.features["img"],
            # while R/rep is only available from model.forward_joint.
            if method_name == "DetBayesRTMMRL":
                rc_feats, available_keys, source, matched_keys, text_proto, text_source = extract_bayesrt_rc_features(method, batch, eval_ctx)
                if class_prototypes is None and text_proto is not None:
                    class_prototypes = text_proto
                    class_prototype_source = str(text_source or "forward_joint.text_features")

                # Save real R/C features first, and keep the exact tensors used for MixFeat.
                det_feats_for_mix = {}
                for branch in [b for b in branches if b in {"R", "C"}]:
                    feat = rc_feats.get(branch)
                    if feat is not None:
                        matched_key = matched_keys.get(branch, f"forward_joint.{branch}")
                        feat = normalize_feature_tensor_shape(feat)
                        if feat is not None:
                            features_by_branch[branch].append(feat)
                            det_feats_for_mix[branch] = feat
                        feature_key_report[branch]["matched_keys"].append(matched_key)
                        if batch_idx < 3:
                            feature_key_report[branch]["available_keys_examples"].append(available_keys)
                    else:
                        # Fallback: try MethodOutputs.features for C/main in case direct path fails.
                        feat_dict = extract_feature_dict_from_outputs(outputs)
                        feat2, matched_key2, available2 = pick_feature_from_dict(feat_dict, branch)
                        feat2 = normalize_feature_tensor_shape(feat2)
                        if feat2 is not None:
                            features_by_branch[branch].append(feat2)
                            det_feats_for_mix[branch] = feat2
                        if matched_key2 is not None:
                            feature_key_report[branch]["matched_keys"].append(str(matched_key2))
                        if batch_idx < 3:
                            feature_key_report[branch]["available_keys_examples"].append([str(x) for x in available2] + [f"bayesrt_direct_status={source}"])

                # Construct Det-MixFeat from the same C/R feature tensors saved above.
                # Feature-level fusion sequence:
                #   C_unit = unit(C); R_unit = unit(R)
                #   blend = alpha * C_unit + (1-alpha) * R_unit
                #   MixFeat = unit(blend)
                if "MixFeat" in branches:
                    alpha = get_bayesrt_alpha(method, default=0.5)
                    mix_feat = build_det_mixfeat_tensor(
                        det_feats_for_mix.get("C"),
                        det_feats_for_mix.get("R"),
                        alpha=alpha,
                    )
                    if mix_feat is not None:
                        features_by_branch["MixFeat"].append(mix_feat)
                        feature_key_report["MixFeat"]["matched_keys"].append(
                            f"constructed_from_saved_C_R_features_alpha={alpha:g}"
                        )
                        if batch_idx < 3:
                            feature_key_report["MixFeat"]["available_keys_examples"].append(
                                [str(x) for x in available_keys]
                                + [
                                    "MixFeat_formula=C_unit=unit(C);R_unit=unit(R);"
                                    f"blend=alpha*C_unit+(1-alpha)*R_unit;MixFeat=unit(blend);alpha={alpha:g}"
                                ]
                            )
                    elif batch_idx < 3:
                        feature_key_report["MixFeat"]["available_keys_examples"].append(
                            [str(x) for x in available_keys] + [f"MixFeat_failed_status={source}"]
                        )

                if class_prototypes is None:
                    proto, proto_source = extract_runtime_class_prototypes(method, method_name=method_name, outputs=outputs)
                    if proto is not None:
                        class_prototypes = proto
                        class_prototype_source = str(proto_source)
            else:
                if class_prototypes is None:
                    proto, proto_source = extract_runtime_class_prototypes(method, method_name=method_name, outputs=outputs)
                    if proto is not None:
                        class_prototypes = proto
                        class_prototype_source = str(proto_source)
                feat_dict = extract_feature_dict_from_outputs(outputs)
                for branch in branches:
                    feat, matched_key, available_keys = pick_feature_from_dict(feat_dict, branch)
                    feat = normalize_feature_tensor_shape(feat)
                    if feat is not None:
                        features_by_branch[branch].append(feat)
                    if matched_key is not None:
                        feature_key_report[branch]["matched_keys"].append(str(matched_key))
                    if batch_idx < 3:
                        feature_key_report[branch]["available_keys_examples"].append([str(x) for x in available_keys])

    logits = torch.cat(logits_all, dim=0)
    labels = torch.cat(labels_all, dim=0) if labels_all else None
    preds = top1_pred_from_logits(logits)
    msp_conf = msp_confidence_from_logits(logits)
    msp_unc = 1.0 - msp_conf
    pred_entropy = predictive_entropy_from_logits(logits)
    correct = (preds == labels).detach().cpu() if labels is not None else None

    features = {}
    for branch, chunks in features_by_branch.items():
        if chunks:
            try:
                features[branch] = torch.cat(chunks, dim=0)
            except Exception as e:
                print(f"[WARN] failed to concat features: method={method_name} branch={branch} error={repr(e)}")

    # compress report
    for branch, rep in feature_key_report.items():
        rep["matched_keys"] = sorted(set(rep["matched_keys"]))
        # avoid huge CSV cells while keeping early-batch diagnostics
        rep["available_keys_examples"] = rep["available_keys_examples"][:3]

    return {
        "logits": logits,
        "labels": labels,
        "preds": preds,
        "msp_confidence": msp_conf,
        "msp_uncertainty": msp_unc.detach().float().cpu(),
        "predictive_entropy": pred_entropy,
        "correct": correct,
        "features": features,
        "class_prototypes": class_prototypes,
        "class_prototype_source": class_prototype_source,
        "feature_key_report": feature_key_report,
    }


def save_tensor_payload(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)


def load_tensor_payload(path: Path):
    return torch.load(path, map_location="cpu")


def payload_meta(method_name, id_dataset, shot, seed, split, ood_dataset=None, model_dir=None):
    return {
        "notebook_version": NOTEBOOK_VERSION,
        "method": method_name,
        "id_dataset": id_dataset,
        "ood_dataset": ood_dataset,
        "protocol": PROTOCOL,
        "shot": int(shot),
        "seed": int(seed),
        "backbone": BACKBONE,
        "split": split,
        "model_dir": str(model_dir) if model_dir is not None else "",
        "main_uncertainty_score": MAIN_UNCERTAINTY_SCORE,
        "save_features": bool(SAVE_FEATURES),
        "feature_branches": expected_branches_for_method(method_name),
        "git_head": git_head,
    }




## 7. 单个 method × ID dataset × shot × seed 的执行逻辑

每个 case 先跑一次 ID test set，再依次跑多个 OOD dataset。ID 结果会被复用。



In [9]:

# =========================
# 7. run one case
# =========================
def run_one_case(method_name: str, id_dataset: str, shot: int, seed: int):
    model_dir = build_model_dir(method_name, id_dataset, shot, seed)
    cache_dir = build_case_cache_dir(method_name, id_dataset, shot, seed)
    cache_dir.mkdir(parents=True, exist_ok=True)

    result_csv = cache_dir / "ood_msp_results.csv"
    status_json = cache_dir / "status.json"

    if (not RECOMPUTE_CACHE) and result_csv.exists():
        print(f"[SKIP] {method_name} {id_dataset} shot={shot} seed={seed}: cached {result_csv}")
        return pd.read_csv(result_csv)

    try:
        trainer, model_dir, cache_dir = build_loaded_trainer(method_name, id_dataset, shot, seed)
    except Exception as e:
        row = {
            "method": method_name,
            "id_dataset": id_dataset,
            "shot": int(shot),
            "seed": int(seed),
            "status": "missing_or_failed_build",
            "error": repr(e),
            "model_dir": str(model_dir),
        }
        with status_json.open("w", encoding="utf-8") as f:
            json.dump(row, f, indent=2, ensure_ascii=False)
        print("[FAIL_BUILD]", row)
        return pd.DataFrame([row])

    impl_report = runtime_implementation_report(trainer, method_name, id_dataset, shot, seed, model_dir)
    save_runtime_implementation_report(impl_report, cache_dir)

    print(f"[ID] collect outputs: {method_name} {id_dataset} shot={shot} seed={seed}")
    id_payload_path = cache_dir / "id_test_outputs.pt"
    id_payload = collect_selected_outputs(
        trainer=trainer,
        loader=trainer.test_loader,
        split_name="test",
        method_name=method_name,
        save_features=SAVE_FEATURES,
    )
    id_payload.update({
        "method": method_name,
        "split": "id_test",
        "meta": payload_meta(method_name, id_dataset, shot, seed, "id_test", model_dir=model_dir),
    })
    save_tensor_payload(id_payload_path, id_payload)

    id_unc = id_payload["msp_uncertainty"].detach().cpu().numpy()
    id_conf = id_payload["msp_confidence"].detach().cpu().numpy()
    id_correct = id_payload["correct"].detach().cpu().numpy() if id_payload.get("correct", None) is not None else None

    rows = []
    feature_report_rows = []
    for branch, rep in id_payload.get("feature_key_report", {}).items():
        feature_report_rows.append({
            "method": method_name,
            "id_dataset": id_dataset,
            "ood_dataset": "",
            "shot": int(shot),
            "seed": int(seed),
            "split": "id_test",
            "feature_branch": branch,
            "matched_keys": ";".join(rep.get("matched_keys", [])),
            "available_keys_examples": json.dumps(rep.get("available_keys_examples", []), ensure_ascii=False),
        })

    for ood_dataset in OOD_DATASETS:
        print(f"[OOD] collect outputs: {method_name} ID={id_dataset} OOD={ood_dataset} shot={shot} seed={seed}")
        try:
            registry_name, ood_loader = build_ood_loader(
                cfg=trainer.cfg,
                ood_dataset_key=ood_dataset,
                batch_size=int(OOD_BATCH_SIZE),
                num_workers=int(OOD_NUM_WORKERS),
            )

            ood_payload = collect_selected_outputs(
                trainer=trainer,
                loader=ood_loader,
                split_name=f"ood_{ood_dataset}",
                method_name=method_name,
                save_features=SAVE_FEATURES,
            )
            ood_payload.update({
                "method": method_name,
                "split": "ood",
                "meta": payload_meta(method_name, id_dataset, shot, seed, "ood", ood_dataset=ood_dataset, model_dir=model_dir),
            })

            ood_payload_path = cache_dir / f"ood_{ood_dataset}_outputs.pt"
            save_tensor_payload(ood_payload_path, ood_payload)

            ood_unc = ood_payload["msp_uncertainty"].detach().cpu().numpy()
            metrics = compute_ood_metrics_from_uncertainty(id_unc, ood_unc)

            row = {
                "method": method_name,
                "id_dataset": id_dataset,
                "ood_dataset": ood_dataset,
                "registry_dataset": registry_name,
                "shot": int(shot),
                "seed": int(seed),
                "score_name": "msp_uncertainty",
                "num_id": int(len(id_unc)),
                "num_ood": int(len(ood_unc)),
                "id_msp_uncertainty_mean": float(np.mean(id_unc)),
                "ood_msp_uncertainty_mean": float(np.mean(ood_unc)),
                "id_msp_confidence_mean": float(np.mean(id_conf)),
                "status": "ok",
                "id_payload_path": str(id_payload_path),
                "ood_payload_path": str(ood_payload_path),
                **metrics,
            }
            if id_correct is not None:
                row.update(compute_error_detection_metrics(id_correct, id_unc))

            for branch, rep in ood_payload.get("feature_key_report", {}).items():
                feature_report_rows.append({
                    "method": method_name,
                    "id_dataset": id_dataset,
                    "ood_dataset": ood_dataset,
                    "shot": int(shot),
                    "seed": int(seed),
                    "split": "ood",
                    "feature_branch": branch,
                    "matched_keys": ";".join(rep.get("matched_keys", [])),
                    "available_keys_examples": json.dumps(rep.get("available_keys_examples", []), ensure_ascii=False),
                })

        except Exception as e:
            row = {
                "method": method_name,
                "id_dataset": id_dataset,
                "ood_dataset": ood_dataset,
                "registry_dataset": "",
                "shot": int(shot),
                "seed": int(seed),
                "score_name": "msp_uncertainty",
                "num_id": int(len(id_unc)),
                "num_ood": 0,
                "status": "failed_ood",
                "error": repr(e),
                "id_payload_path": str(id_payload_path),
                "ood_payload_path": "",
            }
            print("[FAIL_OOD]", row)

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(result_csv, index=False)

    if feature_report_rows:
        pd.DataFrame(feature_report_rows).to_csv(cache_dir / "feature_key_report.csv", index=False)

    with status_json.open("w", encoding="utf-8") as f:
        json.dump({
            "method": method_name,
            "id_dataset": id_dataset,
            "shot": int(shot),
            "seed": int(seed),
            "status": "done",
            "result_csv": str(result_csv),
            "model_dir": str(model_dir),
        }, f, indent=2, ensure_ascii=False)

    return df



## 8. 批量执行并保存 raw OOD 结果

建议先把 `SHOTS=[16]`、`SEEDS=[1]` 小范围验证，再跑全量。



In [10]:

# =========================
# 8. 批量执行
# =========================
all_rows = []

for id_dataset in ID_DATASETS:
    for shot in SHOTS:
        for seed in SEEDS:
            for method_name in ["DetBayesRTMMRL", "BayesAdapter"]:
                print("=" * 100)
                print(f"RUN method={method_name} id={id_dataset} shot={shot} seed={seed}")
                df_case = run_one_case(method_name, id_dataset, shot, seed)
                all_rows.append(df_case)

ood_raw = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
raw_csv = SUMMARY_ROOT / "summary_ood_msp_raw.csv"
ood_raw.to_csv(raw_csv, index=False)

print("Saved raw summary:", raw_csv)
display(ood_raw.head(30))



RUN method=DetBayesRTMMRL id=cifar_10 shot=16 seed=1
[INFO] fallback method_config_file: configs/methods/det_bayesrt_mmrl.yaml
Loading trainer: RefactorRunner
Loading dataset: CIFAR_10
Loading preprocessed few-shot data from /root/autodl-tmp/MMRL/DATASETS/cifar10/split_fewshot/shot_16-seed_1.pkl
Building transform_train
+ random resized crop (size=(224, 224), scale=(0.5, 1))
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
---------  --------
Dataset    CIFAR_10
# classes  10
# train_x  160
# val      40
# test     10,000
---------  --------
[BayesRTMMRL] trainable params: {'representation_learner.compound_rep_tokens_r2vproj.3.bias', 'representation_learner.compound_rep_tokens_

/root/autodl-tmp/MMRL/trainers/refactor_runner.py:75: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if prec == "amp" else None
/tmp/ipykernel_10693/3395882264.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[OOD] collect outputs: DetBayesRTMMRL ID=cifar_10 OOD=dtd shot=16 seed=1
Reading split from /root/autodl-tmp/MMRL/DATASETS/dtd/split_zhou_DescribableTextures.json
[OOD] dtd: registry=DescribableTextures, num_test=1692
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
RUN method=BayesAdapter id=cifar_10 shot=16 seed=1
Loading trainer: RefactorRunner
Loading dataset: CIFAR_10
Loading preprocessed few-shot data from /root/autodl-tmp/MMRL/DATASETS/cifar10/split_fewshot/shot_16-seed_1.pkl
Building transform_train
+ random resized crop (size=(224, 224), scale=(0.08, 1.0))
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+

/root/autodl-tmp/MMRL/trainers/refactor_runner.py:75: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if prec == "amp" else None
/tmp/ipykernel_10693/3395882264.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[OOD] collect outputs: BayesAdapter ID=cifar_10 OOD=dtd shot=16 seed=1
Reading split from /root/autodl-tmp/MMRL/DATASETS/dtd/split_zhou_DescribableTextures.json
[OOD] dtd: registry=DescribableTextures, num_test=1692
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
Saved raw summary: /root/autodl-tmp/MMRL/output_refactor/analysis/original_v8_logic_separate_vector/summaries/summary_ood_msp_raw.csv


,method,id_dataset,ood_dataset,registry_dataset,shot,seed,score_name,num_id,num_ood,id_msp_uncertainty_mean,ood_msp_uncertainty_mean,id_msp_confidence_mean,status,id_payload_path,ood_payload_path,AUROC,AUPR_OUT,AUPR_IN,FPR95,DetectionError,ErrorAUROC,ErrorAUPR,ErrorFPR95
0,DetBayesRTMMRL,cifar_10,dtd,DescribableTextures,16,1,msp_uncertainty,10000,1692,0.04511,0.544644,0.95489,ok,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,0.974451,0.889399,0.995011,0.1077,0.078236,0.923764,0.402282,0.303235
1,BayesAdapter,cifar_10,dtd,DescribableTextures,16,1,msp_uncertainty,10000,1692,0.08361,0.516533,0.91639,ok,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,0.944882,0.754198,0.989601,0.2119,0.120508,0.923100,0.471101,0.308871


Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/original_v8_logic_separate_vector/summaries/summary_ood_msp_mean_std.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/original_v8_logic_separate_vector/summaries/summary_ood_msp_delta.csv
ood_mean_std shape: (2, 23)
ood_delta shape: (8, 11)
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/original_v8_logic_separate_vector/summaries/msp_uncertainty_distribution_figures.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/original_v8_logic_separate_vector/summaries/msp_correct_wrong_ood_figures.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/original_v8_logic_separate_vector/summaries/feature_embedding_R_C_MixFeat_adapter_figures.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/original_v8_logic_separate_vector/summaries/feature_sanity_checks.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/original_v8_logic_separate_vector/summaries/similarity_matrix_R_C_MixFeat_adapter_figures.csv
Saved

## 9. mean/std 与 delta 汇总

`delta = DetBayesRTMMRL - BayesAdapter`。

注意指标方向：

- `AUROC`, `AUPR_OUT`, `AUPR_IN`, `ErrorAUROC`, `ErrorAUPR`：越高越好。
- `FPR95`, `DetectionError`, `ErrorFPR95`, `ECE`, `NLL`, `Brier`, `AURC`, `EAURC`：越低越好。



In [11]:
# =========================
# 9. 汇总表
# =========================
HIGHER_BETTER_METRICS = ["AUROC", "AUPR_OUT", "AUPR_IN", "ErrorAUROC", "ErrorAUPR", "Accuracy"]
LOWER_BETTER_METRICS = ["FPR95", "DetectionError", "ErrorFPR95", "ECE", "MCE", "NLL", "Brier", "AURC", "EAURC"]
OOD_METRIC_COLS = ["AUROC", "AUPR_OUT", "AUPR_IN", "FPR95", "DetectionError", "ErrorAUROC", "ErrorAUPR", "ErrorFPR95"]

OOD_MEAN_STD_COLUMNS = ["method", "id_dataset", "ood_dataset", "shot", "score_name", "num_seeds", "seeds"]
for _m in OOD_METRIC_COLS:
    OOD_MEAN_STD_COLUMNS.extend([f"{_m}_mean", f"{_m}_std"])

OOD_DELTA_COLUMNS = [
    "id_dataset", "ood_dataset", "shot", "score_name", "metric", "metric_direction",
    "DetBayesRTMMRL", "BayesAdapter", "delta_DetBayesRTMMRL_minus_BayesAdapter",
    "better_method", "status",
]


def metric_direction(metric):
    if metric in HIGHER_BETTER_METRICS:
        return "higher_better"
    if metric in LOWER_BETTER_METRICS:
        return "lower_better"
    return "unknown"


def better_method_from_delta(delta, metric):
    if pd.isna(delta):
        return "unknown"
    direction = metric_direction(metric)
    if direction == "higher_better":
        return "DetBayesRTMMRL" if delta > 0 else ("BayesAdapter" if delta < 0 else "tie")
    if direction == "lower_better":
        return "DetBayesRTMMRL" if delta < 0 else ("BayesAdapter" if delta > 0 else "tie")
    return "unknown"


def empty_ood_mean_std_df():
    return pd.DataFrame(columns=OOD_MEAN_STD_COLUMNS)


def empty_ood_delta_df():
    return pd.DataFrame(columns=OOD_DELTA_COLUMNS)


def filter_ok_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only status == ok rows. If status is absent, treat all rows as ok."""
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    if "status" not in out.columns:
        return out
    status = out["status"].fillna("ok").astype(str)
    return out[status.eq("ok")].copy()


def mean_std_summary(df: pd.DataFrame, metric_cols):
    ok = filter_ok_rows(df)
    if ok.empty:
        return empty_ood_mean_std_df()

    required_cols = ["method", "id_dataset", "ood_dataset", "shot", "score_name", "seed"]
    missing = [c for c in required_cols if c not in ok.columns]
    if missing:
        print(f"[WARN] mean_std_summary missing required columns: {missing}")
        return empty_ood_mean_std_df()

    group_cols = ["method", "id_dataset", "ood_dataset", "shot", "score_name"]
    rows = []
    for keys, g in ok.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys))
        seeds = pd.to_numeric(g["seed"], errors="coerce").dropna().astype(int).unique()
        seeds = sorted(seeds.tolist())
        row["num_seeds"] = int(len(seeds))
        row["seeds"] = " ".join(str(x) for x in seeds)

        for m in metric_cols:
            vals = pd.to_numeric(g[m], errors="coerce").dropna() if m in g.columns else pd.Series(dtype=float)
            row[f"{m}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{m}_std"] = float(vals.std(ddof=0)) if len(vals) > 1 else (0.0 if len(vals) == 1 else np.nan)
        rows.append(row)

    if not rows:
        return empty_ood_mean_std_df()
    return pd.DataFrame(rows, columns=OOD_MEAN_STD_COLUMNS).sort_values(
        ["id_dataset", "ood_dataset", "shot", "method", "score_name"],
        kind="mergesort",
    ).reset_index(drop=True)


def delta_summary(df_mean: pd.DataFrame, metric_cols):
    """Compare DetBayesRTMMRL and BayesAdapter under the same score_name.

    Current OOD rows use score_name='msp_uncertainty' for both methods, so this
    preserves the original comparison口径. The function is now robust to missing
    columns and always returns a well-formed DataFrame.
    """
    if df_mean is None or df_mean.empty:
        return empty_ood_delta_df()

    required_cols = ["method", "id_dataset", "ood_dataset", "shot", "score_name"]
    missing = [c for c in required_cols if c not in df_mean.columns]
    if missing:
        print(f"[WARN] delta_summary missing required columns: {missing}")
        return empty_ood_delta_df()

    id_cols = ["id_dataset", "ood_dataset", "shot", "score_name"]
    rows = []
    for keys, g in df_mean.groupby(id_cols, dropna=False):
        base = dict(zip(id_cols, keys))
        rt = g[g["method"].astype(str).eq("DetBayesRTMMRL")]
        ba = g[g["method"].astype(str).eq("BayesAdapter")]
        if rt.empty or ba.empty:
            rows.append({**base, "metric": np.nan, "status": "missing_pair"})
            continue

        # If duplicated rows exist for a method/score, use the first sorted row but do not silently crash.
        rt_row = rt.iloc[0]
        ba_row = ba.iloc[0]
        for m in metric_cols:
            rt_val = pd.to_numeric(rt_row.get(f"{m}_mean", np.nan), errors="coerce")
            ba_val = pd.to_numeric(ba_row.get(f"{m}_mean", np.nan), errors="coerce")
            if pd.isna(rt_val) or pd.isna(ba_val):
                delta = np.nan
                status = "missing_metric"
            else:
                rt_val = float(rt_val)
                ba_val = float(ba_val)
                delta = rt_val - ba_val
                status = "ok"
            rows.append({
                **base,
                "metric": m,
                "metric_direction": metric_direction(m),
                "DetBayesRTMMRL": rt_val,
                "BayesAdapter": ba_val,
                "delta_DetBayesRTMMRL_minus_BayesAdapter": delta,
                "better_method": better_method_from_delta(delta, m),
                "status": status,
            })

    if not rows:
        return empty_ood_delta_df()
    return pd.DataFrame(rows, columns=OOD_DELTA_COLUMNS).sort_values(
        ["id_dataset", "ood_dataset", "shot", "score_name", "metric"],
        kind="mergesort",
    ).reset_index(drop=True)


ood_mean_std = mean_std_summary(ood_raw, OOD_METRIC_COLS)
ood_mean_std_csv = SUMMARY_ROOT / "summary_ood_msp_mean_std.csv"
ood_mean_std.to_csv(ood_mean_std_csv, index=False)

ood_delta = delta_summary(ood_mean_std, OOD_METRIC_COLS)
ood_delta_csv = SUMMARY_ROOT / "summary_ood_msp_delta.csv"
ood_delta.to_csv(ood_delta_csv, index=False)

print("Saved:", ood_mean_std_csv)
print("Saved:", ood_delta_csv)
print("ood_mean_std shape:", ood_mean_std.shape)
print("ood_delta shape:", ood_delta.shape)
display(ood_mean_std.head(30))
display(ood_delta.head(30))



,method,id_dataset,ood_dataset,shot,score_name,num_seeds,seeds,AUROC_mean,AUROC_std,AUPR_OUT_mean,AUPR_OUT_std,AUPR_IN_mean,AUPR_IN_std,FPR95_mean,FPR95_std,DetectionError_mean,DetectionError_std,ErrorAUROC_mean,ErrorAUROC_std,ErrorAUPR_mean,ErrorAUPR_std,ErrorFPR95_mean,ErrorFPR95_std
0,BayesAdapter,cifar_10,dtd,16,msp_uncertainty,1,1,0.944882,0.0,0.754198,0.0,0.989601,0.0,0.2119,0.0,0.120508,0.0,0.923100,0.0,0.471101,0.0,0.308871,0.0
1,DetBayesRTMMRL,cifar_10,dtd,16,msp_uncertainty,1,1,0.974451,0.0,0.889399,0.0,0.995011,0.0,0.1077,0.0,0.078236,0.0,0.923764,0.0,0.402282,0.0,0.303235,0.0


,id_dataset,ood_dataset,shot,score_name,metric,metric_direction,DetBayesRTMMRL,BayesAdapter,delta_DetBayesRTMMRL_minus_BayesAdapter,better_method,status
0,cifar_10,dtd,16,msp_uncertainty,AUPR_IN,higher_better,0.995011,0.989601,0.005409,DetBayesRTMMRL,ok
1,cifar_10,dtd,16,msp_uncertainty,AUPR_OUT,higher_better,0.889399,0.754198,0.135201,DetBayesRTMMRL,ok
2,cifar_10,dtd,16,msp_uncertainty,AUROC,higher_better,0.974451,0.944882,0.029570,DetBayesRTMMRL,ok
3,cifar_10,dtd,16,msp_uncertainty,DetectionError,lower_better,0.078236,0.120508,-0.042272,DetBayesRTMMRL,ok
4,cifar_10,dtd,16,msp_uncertainty,ErrorAUPR,higher_better,0.402282,0.471101,-0.068819,BayesAdapter,ok
5,cifar_10,dtd,16,msp_uncertainty,ErrorAUROC,higher_better,0.923764,0.923100,0.000664,DetBayesRTMMRL,ok
6,cifar_10,dtd,16,msp_uncertainty,ErrorFPR95,lower_better,0.303235,0.308871,-0.005635,DetBayesRTMMRL,ok
7,cifar_10,dtd,16,msp_uncertainty,FPR95,lower_better,0.107700,0.211900,-0.104200,DetBayesRTMMRL,ok


## 10. 绘图工具：KDE / ECDF 曲线

分布图只能用曲线，不画柱状图。



In [12]:

# =========================
# 10. 绘图工具
# =========================
import matplotlib.pyplot as plt


# Vector-output settings only. These do not change the analysis or data.
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


def as_axes_array(axes):
    """Return axes as a flat object array, robust for 1D/2D/single-axis cases."""
    return np.asarray(axes, dtype=object).reshape(-1)


def make_axes_square(axes):
    """
    Make each subplot physically square without changing data values.
    Uses matplotlib's set_box_aspect when available.
    """
    for ax in as_axes_array(axes):
        if ax is None:
            continue
        try:
            ax.set_box_aspect(1)
        except Exception:
            # Older matplotlib: leave data aspect unchanged rather than forcing equal
            # data units, because distribution curves should not be distorted.
            pass
    return axes


def square_grid_figsize(nrows, ncols, panel_size=None, extra_width=0.0, extra_height=0.0):
    panel_size = float(FIG_PANEL_SIZE if panel_size is None else panel_size)
    return (
        max(panel_size * int(ncols) + float(extra_width), panel_size),
        max(panel_size * int(nrows) + float(extra_height), panel_size),
    )


def to_numpy_1d(x):
    if x is None:
        return np.array([], dtype=float)
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy().reshape(-1)
    return np.asarray(x).reshape(-1)


def density_curve(values, x_grid=None, value_range=(0.0, 1.0), num=256):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if x_grid is None:
        x_grid = np.linspace(value_range[0], value_range[1], num)
    if len(vals) < 2:
        return x_grid, np.zeros_like(x_grid)
    if SCIPY_AVAILABLE:
        try:
            kde = gaussian_kde(vals)
            y = kde(x_grid)
            return x_grid, y
        except Exception:
            pass
    # Fallback: line-smoothed histogram density, still plotted as a curve.
    counts, edges = np.histogram(vals, bins=min(80, max(10, int(np.sqrt(len(vals))))), range=value_range, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    y = np.interp(x_grid, centers, counts, left=0.0, right=0.0)
    kernel = np.ones(5) / 5
    y = np.convolve(y, kernel, mode="same")
    return x_grid, y


def ecdf_curve(values):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    vals = np.sort(vals)
    if len(vals) == 0:
        return vals, vals
    y = np.arange(1, len(vals) + 1) / len(vals)
    return vals, y


def plot_distribution_line(ax, values, label, mode="KDE", value_range=(0.0, 1.0), linestyle="-", linewidth=1.8):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return
    if str(mode).upper() == "ECDF":
        x, y = ecdf_curve(vals)
        ax.plot(x, y, label=label, linestyle=linestyle, linewidth=linewidth)
        ax.set_ylabel("ECDF")
    else:
        x, y = density_curve(vals, value_range=value_range)
        ax.plot(x, y, label=label, linestyle=linestyle, linewidth=linewidth)
        ax.set_ylabel("Density")


def load_case_payloads(method, id_dataset, shot, seed, ood_dataset):
    cache_dir = build_case_cache_dir(method, id_dataset, shot, seed)
    id_path = cache_dir / "id_test_outputs.pt"
    ood_path = cache_dir / f"ood_{ood_dataset}_outputs.pt"
    if not id_path.exists() or not ood_path.exists():
        return None, None
    return load_tensor_payload(id_path), load_tensor_payload(ood_path)


def get_unc(payload):
    return to_numpy_1d(payload.get("msp_uncertainty", None))


def get_labels(payload):
    labels = payload.get("labels", None)
    return None if labels is None else to_numpy_1d(labels).astype(int)


def get_correct(payload):
    corr = payload.get("correct", None)
    return None if corr is None else to_numpy_1d(corr).astype(bool)


## 11. 主图：MSP uncertainty ID/OOD 分布曲线

每张 figure 固定：`ID dataset × OOD dataset × shot`。

每个 seed 是一个 subplot；每个 subplot 中同时画：

- `DetBayesRTMMRL - ID`
- `DetBayesRTMMRL - OOD`
- `BayesAdapter - ID`
- `BayesAdapter - OOD`



In [13]:
# =========================
# 11. MSP uncertainty ID/OOD seed panels
# =========================
def plot_msp_uncertainty_seed_panels(id_dataset, ood_dataset, shot, mode="KDE"):
    nrows, ncols = 1, len(SEEDS)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=square_grid_figsize(nrows, ncols, panel_size=FIG_PANEL_SIZE, extra_height=FIG_TOP_EXTRA_HEIGHT),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    make_axes_square(axes)

    any_plotted = False
    for ax, seed in zip(axes.reshape(-1), SEEDS):
        for method, linestyle in [("DetBayesRTMMRL", "-"), ("BayesAdapter", "--")]:
            id_payload, ood_payload = load_case_payloads(method, id_dataset, shot, seed, ood_dataset)
            if id_payload is None or ood_payload is None:
                continue
            id_unc = get_unc(id_payload)
            ood_unc = get_unc(ood_payload)
            plot_distribution_line(ax, id_unc, f"{method} ID", mode=mode, linestyle=linestyle)
            plot_distribution_line(ax, ood_unc, f"{method} OOD", mode=mode, linestyle=linestyle)
            any_plotted = True

        ax.set_title(f"seed={seed}", fontsize=11)
        ax.set_xlabel("MSP uncertainty = 1 - max softmax probability")
        ax.set_xlim(0.0, 1.0)
        ax.grid(alpha=0.25)

    axes.reshape(-1)[0].legend(fontsize=9)
    fig.suptitle(f"MSP uncertainty distribution: {id_dataset} vs {ood_dataset}, shot={shot}", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    if not any_plotted:
        plt.close(fig)
        return None

    out_dir = FIG_DIRS["msp_kde"] if str(mode).upper() == "KDE" else FIG_DIRS["msp_ecdf"]
    out_path = out_dir / f"msp_uncertainty_{str(mode).lower()}_{id_dataset}_{ood_dataset}_shot{shot}_seed_panels_square_v7.png"
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return str(out_path)

msp_fig_rows = []
for id_dataset in ID_DATASETS:
    for ood_dataset in OOD_DATASETS:
        for shot in SHOTS:
            p = plot_msp_uncertainty_seed_panels(id_dataset, ood_dataset, shot, mode=DISTRIBUTION_PLOT_TYPE)
            if p:
                msp_fig_rows.append({"id_dataset": id_dataset, "ood_dataset": ood_dataset, "shot": shot, "mode": DISTRIBUTION_PLOT_TYPE, "figure": p})

msp_fig_df = pd.DataFrame(msp_fig_rows)
msp_fig_csv = SUMMARY_ROOT / "msp_uncertainty_distribution_figures.csv"
msp_fig_df.to_csv(msp_fig_csv, index=False)
print("Saved:", msp_fig_csv)
display(msp_fig_df.head(20))


,id_dataset,ood_dataset,shot,mode,figure
0,cifar_10,dtd,16,KDE,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 12. ID correct / ID wrong / OOD 的 MSP uncertainty 曲线

每张 figure 固定：`ID dataset × OOD dataset × shot`。

布局：

```text
rows = seeds
columns = methods
```

每个 panel 中画三条曲线：`ID correct`、`ID wrong`、`OOD`。



In [14]:
# =========================
# 12. Correct / Wrong / OOD uncertainty panels
# =========================
def plot_correct_wrong_ood_panels(id_dataset, ood_dataset, shot, mode="KDE"):
    methods = ["DetBayesRTMMRL", "BayesAdapter"]
    nrows, ncols = len(SEEDS), len(methods)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=square_grid_figsize(nrows, ncols, panel_size=FIG_PANEL_SIZE, extra_height=FIG_TOP_EXTRA_HEIGHT),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    make_axes_square(axes)

    any_plotted = False
    for r, seed in enumerate(SEEDS):
        for c, method in enumerate(methods):
            ax = axes[r, c]
            id_payload, ood_payload = load_case_payloads(method, id_dataset, shot, seed, ood_dataset)
            if id_payload is not None and ood_payload is not None:
                id_unc = get_unc(id_payload)
                ood_unc = get_unc(ood_payload)
                correct = get_correct(id_payload)
                if correct is not None and len(correct) == len(id_unc):
                    plot_distribution_line(ax, id_unc[correct], "ID correct", mode=mode, linestyle="-")
                    plot_distribution_line(ax, id_unc[~correct], "ID wrong", mode=mode, linestyle="--")
                else:
                    plot_distribution_line(ax, id_unc, "ID", mode=mode, linestyle="-")
                plot_distribution_line(ax, ood_unc, "OOD", mode=mode, linestyle=":")
                any_plotted = True
            ax.set_title(f"{method}, seed={seed}", fontsize=11)
            ax.set_xlabel("MSP uncertainty")
            ax.set_xlim(0.0, 1.0)
            ax.grid(alpha=0.25)
            if r == 0 and c == 0:
                ax.legend(fontsize=9)

    fig.suptitle(f"ID correct / ID wrong / OOD MSP uncertainty: {id_dataset} vs {ood_dataset}, shot={shot}", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    if not any_plotted:
        plt.close(fig)
        return None
    out_path = FIG_DIRS["msp_cwo"] / f"msp_correct_wrong_ood_{id_dataset}_{ood_dataset}_shot{shot}_seed_method_panels_square_v7.png"
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return str(out_path)

cwo_fig_rows = []
for id_dataset in ID_DATASETS:
    for ood_dataset in OOD_DATASETS:
        for shot in SHOTS:
            p = plot_correct_wrong_ood_panels(id_dataset, ood_dataset, shot, mode=DISTRIBUTION_PLOT_TYPE)
            if p:
                cwo_fig_rows.append({"id_dataset": id_dataset, "ood_dataset": ood_dataset, "shot": shot, "mode": DISTRIBUTION_PLOT_TYPE, "figure": p})

cwo_fig_df = pd.DataFrame(cwo_fig_rows)
cwo_fig_csv = SUMMARY_ROOT / "msp_correct_wrong_ood_figures.csv"
cwo_fig_df.to_csv(cwo_fig_csv, index=False)
print("Saved:", cwo_fig_csv)
display(cwo_fig_df.head(20))


,id_dataset,ood_dataset,shot,mode,figure
0,cifar_10,dtd,16,KDE,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 13. 特征采样与 embedding 工具

DetBayesRTMMRL 需要分别画：

- `R branch`
- `C branch`

BayesAdapter 画：

- `adapter feature`

每个 seed 单独导出一张图，列布局为：

```text
DetBayesRTMMRL-R | DetBayesRTMMRL-C | BayesAdapter
```



In [15]:

# =========================
# 13. 特征采样与 embedding 工具
# =========================
def get_feature_np(payload, branch):
    feats = payload.get("features", {})
    if not isinstance(feats, dict) or branch not in feats:
        return None
    x = feats[branch]
    if isinstance(x, torch.Tensor):
        return x.detach().float().cpu().numpy()
    return np.asarray(x, dtype=np.float32)


def sample_id_indices_by_class(labels, max_per_class=80, random_state=2026):
    rng = np.random.default_rng(random_state)
    labels = np.asarray(labels).astype(int)
    indices = []
    for y in sorted(np.unique(labels)):
        idx = np.where(labels == y)[0]
        if len(idx) > max_per_class:
            idx = rng.choice(idx, size=max_per_class, replace=False)
        indices.extend(idx.tolist())
    return np.asarray(sorted(indices), dtype=int)


def sample_ood_indices(n, max_total=800, random_state=2026):
    rng = np.random.default_rng(random_state)
    idx = np.arange(n)
    if n > max_total:
        idx = rng.choice(idx, size=max_total, replace=False)
    return np.asarray(sorted(idx), dtype=int)


def compute_2d_embedding(x, method="umap", random_state=2026):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x)
    n = len(x)
    if n == 0:
        return np.zeros((0, 2), dtype=np.float32)

    # PCA 预降维，提升 t-SNE/UMAP 稳定性。
    if SKLEARN_EMBED_AVAILABLE and x.shape[1] > 50:
        x_reduced = PCA(n_components=50, random_state=random_state).fit_transform(x)
    else:
        x_reduced = x

    if method.lower() == "umap" and UMAP_AVAILABLE:
        reducer = umap.UMAP(n_components=2, random_state=random_state, n_neighbors=15, min_dist=0.1, metric="cosine")
        return reducer.fit_transform(x_reduced)

    if method.lower() == "tsne" and SKLEARN_EMBED_AVAILABLE:
        perplexity = min(30, max(5, (n - 1) // 3))
        return TSNE(n_components=2, random_state=random_state, init="pca", learning_rate="auto", perplexity=perplexity).fit_transform(x_reduced)

    if SKLEARN_EMBED_AVAILABLE:
        return PCA(n_components=2, random_state=random_state).fit_transform(x_reduced)

    # Last-resort fallback: first two dimensions.
    if x.shape[1] >= 2:
        return x[:, :2]
    return np.c_[x[:, 0], np.zeros(n)]


def build_embedding_data(id_payload, ood_payload, branch, random_state=2026):
    x_id = get_feature_np(id_payload, branch)
    x_ood = get_feature_np(ood_payload, branch)
    if x_id is None or x_ood is None:
        return None
    x_id = np.asarray(x_id, dtype=np.float32)
    x_ood = np.asarray(x_ood, dtype=np.float32)

    labels = get_labels(id_payload)
    if labels is None:
        labels = np.zeros(len(x_id), dtype=int)
    labels = np.asarray(labels).astype(int)

    # Guard against feature/label mismatch before indexing. This should not happen
    # for correctly extracted features; trimming here prevents plotting code from
    # crashing while the earlier feature_sanity_checks.csv exposes the mismatch.
    n_id = min(len(x_id), len(labels))
    if n_id == 0 or len(x_ood) == 0:
        return None
    if n_id < len(x_id) or n_id < len(labels):
        print(f"[WARN] embedding feature/label length mismatch branch={branch}: features={len(x_id)} labels={len(labels)}; trimming to {n_id}")
    x_id = x_id[:n_id]
    labels = labels[:n_id]

    id_idx = sample_id_indices_by_class(labels, MAX_ID_PER_CLASS_FOR_FEATURE, random_state=random_state)
    ood_idx = sample_ood_indices(len(x_ood), MAX_OOD_FOR_FEATURE, random_state=random_state)
    if len(id_idx) == 0 or len(ood_idx) == 0:
        return None

    x = np.vstack([x_id[id_idx], x_ood[ood_idx]])
    domain = np.array(["ID"] * len(id_idx) + ["OOD"] * len(ood_idx))
    y = np.r_[labels[id_idx], np.array([-1] * len(ood_idx))]
    emb = compute_2d_embedding(l2_normalize_np(x), method="umap" if UMAP_AVAILABLE else "tsne", random_state=random_state)
    return {"emb": emb, "domain": domain, "labels": y}



## 14. 特征 embedding 图：R / C / MixFeat / Adapter

每个 seed 单独生成一张图。



In [16]:
# =========================
# 14. Feature embedding plots
# =========================
def plot_feature_embedding_rc_adapter(id_dataset, ood_dataset, shot, seed):
    panels = [
        ("DetBayesRTMMRL", "R", "DetBayesRTMMRL - R branch"),
        ("DetBayesRTMMRL", "C", "DetBayesRTMMRL - C branch"),
        ("DetBayesRTMMRL", "MixFeat", "DetBayesRTMMRL - MixFeat"),
        ("BayesAdapter", "adapter", "BayesAdapter"),
    ]
    nrows, ncols = 1, len(panels)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=square_grid_figsize(nrows, ncols, panel_size=FIG_PANEL_SIZE, extra_width=1.7, extra_height=FIG_TOP_EXTRA_HEIGHT),
        sharex=False,
        sharey=False,
        squeeze=False,
    )
    axes = axes.reshape(-1)
    make_axes_square(axes)
    any_plotted = False

    for ax, (method, branch, title) in zip(axes, panels):
        id_payload, ood_payload = load_case_payloads(method, id_dataset, shot, seed, ood_dataset)
        if id_payload is None or ood_payload is None:
            ax.set_title(title + "\nmissing payload", fontsize=11)
            ax.axis("off")
            continue
        data = build_embedding_data(id_payload, ood_payload, branch, random_state=RANDOM_STATE + int(seed))
        if data is None:
            ax.set_title(title + "\nmissing feature", fontsize=11)
            ax.axis("off")
            continue

        emb = data["emb"]
        domain = data["domain"]
        labels = data["labels"]
        id_mask = domain == "ID"
        ood_mask = domain == "OOD"

        # ID by class. 不指定具体颜色，使用 matplotlib 默认循环。
        for y in sorted(np.unique(labels[id_mask])):
            m = id_mask & (labels == y)
            ax.scatter(emb[m, 0], emb[m, 1], s=12, alpha=0.68, label=f"ID {y}")
        ax.scatter(emb[ood_mask, 0], emb[ood_mask, 1], s=16, alpha=0.78, marker="x", label="OOD")
        ax.set_title(title, fontsize=11)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(alpha=0.12)
        any_plotted = True

    # 只放一个简化图例，避免压缩子图；bbox 预留右侧空间。
    handles, labels_ = axes[-1].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels_, loc="center right", fontsize=9, frameon=False)
    fig.suptitle(f"Feature embedding: {id_dataset} vs {ood_dataset}, shot={shot}, seed={seed}", fontsize=13)
    fig.tight_layout(rect=[0, 0, 0.935, 0.94])

    if not any_plotted:
        plt.close(fig)
        return None
    out_path = FIG_DIRS["feature_embedding"] / f"embedding_{id_dataset}_{ood_dataset}_shot{shot}_seed{seed}_R_C_MixFeat_adapter_square_v7.png"
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return str(out_path)

embedding_fig_rows = []
for id_dataset in ID_DATASETS:
    for ood_dataset in OOD_DATASETS:
        for shot in SHOTS:
            for seed in SEEDS:
                p = plot_feature_embedding_rc_adapter(id_dataset, ood_dataset, shot, seed)
                if p:
                    embedding_fig_rows.append({"id_dataset": id_dataset, "ood_dataset": ood_dataset, "shot": shot, "seed": seed, "figure": p})

embedding_fig_df = pd.DataFrame(embedding_fig_rows)
embedding_fig_csv = SUMMARY_ROOT / "feature_embedding_R_C_MixFeat_adapter_figures.csv"
embedding_fig_df.to_csv(embedding_fig_csv, index=False)
print("Saved:", embedding_fig_csv)
display(embedding_fig_df.head(20))


,id_dataset,ood_dataset,shot,seed,figure
0,cifar_10,dtd,16,1,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 15. Prototype-level 特征相似度矩阵与统计（含 Det-MixFeat）

每个 seed 单独一张 figure。这里不再展示 sample-sample 全矩阵，因为矩阵过大且难以比较。

新的 heatmap 全部基于 prototype：

```text
                 ID class-class     OOD class-class     ID class-OOD dataset
DetBayesRTMMRL-R        10 x 10        K_ood x K_ood       10 x K_ood
DetBayesRTMMRL-C        10 x 10        K_ood x K_ood       10 x K_ood
DetBayesRTMMRL-MixFeat  10 x 10        K_ood x K_ood       10 x K_ood
BayesAdapter            10 x 10        K_ood x K_ood       10 x K_ood
```

采样规则：

```text
ID:  每个 CIFAR-10 class 取 PROTOTYPE_ID_PER_CLASS=20 个样本。
OOD: 每个 OOD class 取 PROTOTYPE_OOD_PER_CLASS=20 个样本；若 OOD label 不可用，则整体视为一个 OOD group。
```

ID-OOD 不使用 `10 × OOD类别数`，而使用 `10 × 1`：10 个 ID class prototype 对一个 OOD dataset prototype。这样不同 OOD 数据集之间可比较。




In [17]:
# =========================
# 14b. Feature sanity checks before similarity analysis
# =========================
def _feature_hash_np(x, max_rows=32):
    import hashlib
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2 or len(x) == 0:
        return ""
    sample = np.ascontiguousarray(x[:min(len(x), max_rows)])
    return hashlib.sha256(sample.tobytes()).hexdigest()[:16]


def summarize_feature_payload(payload, branch):
    x = get_feature_np(payload, branch)
    if x is None:
        return {"exists": False}
    x = np.asarray(x, dtype=np.float32)
    norms = np.linalg.norm(x.reshape(len(x), -1), axis=1) if len(x) else np.asarray([])
    return {
        "exists": True,
        "shape": str(tuple(x.shape)),
        "mean_norm": float(np.mean(norms)) if len(norms) else np.nan,
        "std_norm": float(np.std(norms)) if len(norms) else np.nan,
        "feature_hash16": _feature_hash_np(x),
    }


def cosine_between_aligned_features(payload_a, branch_a, payload_b, branch_b, max_rows=512):
    xa = get_feature_np(payload_a, branch_a)
    xb = get_feature_np(payload_b, branch_b)
    if xa is None or xb is None:
        return np.nan
    xa = np.asarray(xa, dtype=np.float32)
    xb = np.asarray(xb, dtype=np.float32)
    n = min(len(xa), len(xb), int(max_rows))
    if n == 0 or xa.shape[1:] != xb.shape[1:]:
        return np.nan
    xa = l2_normalize_np(xa[:n].reshape(n, -1))
    xb = l2_normalize_np(xb[:n].reshape(n, -1))
    return float(np.mean(np.sum(xa * xb, axis=1)))


sanity_rows = []
for id_dataset in ID_DATASETS:
    for shot in SHOTS:
        for seed in SEEDS:
            payloads = {}
            for method, branches in FEATURE_BRANCHES.items():
                cache_dir = build_case_cache_dir(method, id_dataset, shot, seed)
                id_path = cache_dir / "id_test_outputs.pt"
                if id_path.exists():
                    payloads[method] = load_tensor_payload(id_path)
                    for branch in branches:
                        row = {
                            "method": method,
                            "feature_branch": branch,
                            "id_dataset": id_dataset,
                            "shot": int(shot),
                            "seed": int(seed),
                        }
                        row.update(summarize_feature_payload(payloads[method], branch))
                        sanity_rows.append(row)
            if "DetBayesRTMMRL" in payloads:
                p = payloads["DetBayesRTMMRL"]
                sanity_rows.append({
                    "method": "DetBayesRTMMRL",
                    "feature_branch": "R_vs_C_aligned_cosine",
                    "id_dataset": id_dataset,
                    "shot": int(shot),
                    "seed": int(seed),
                    "exists": True,
                    "shape": "",
                    "mean_norm": cosine_between_aligned_features(p, "R", p, "C"),
                    "std_norm": np.nan,
                    "feature_hash16": "diagnostic",
                })
            if "DetBayesRTMMRL" in payloads and "BayesAdapter" in payloads:
                sanity_rows.append({
                    "method": "DetBayesRTMMRL_vs_BayesAdapter",
                    "feature_branch": "C_vs_adapter_aligned_cosine",
                    "id_dataset": id_dataset,
                    "shot": int(shot),
                    "seed": int(seed),
                    "exists": True,
                    "shape": "",
                    "mean_norm": cosine_between_aligned_features(payloads["DetBayesRTMMRL"], "C", payloads["BayesAdapter"], "adapter"),
                    "std_norm": np.nan,
                    "feature_hash16": "diagnostic",
                })

feature_sanity_df = pd.DataFrame(sanity_rows)
feature_sanity_csv = SUMMARY_ROOT / "feature_sanity_checks.csv"
feature_sanity_df.to_csv(feature_sanity_csv, index=False)
print("Saved:", feature_sanity_csv)
display(feature_sanity_df.head(50))



,method,feature_branch,id_dataset,shot,seed,exists,shape,mean_norm,std_norm,feature_hash16
0,DetBayesRTMMRL,R,cifar_10,16,1,True,"(10000, 512)",1.000000,3.942927e-08,1fd09785a7eebfe6
1,DetBayesRTMMRL,C,cifar_10,16,1,True,"(10000, 512)",1.000000,4.858784e-08,aba110d730259016
2,DetBayesRTMMRL,MixFeat,cifar_10,16,1,True,"(10000, 512)",1.000000,5.190740e-08,b5707c6f5f760686
3,BayesAdapter,adapter,cifar_10,16,1,True,"(10000, 512)",10.902704,3.909884e-01,d7ba8f40f69781e9
4,DetBayesRTMMRL,R_vs_C_aligned_cosine,cifar_10,16,1,True,,0.107077,NaN,diagnostic
5,DetBayesRTMMRL_vs_BayesAdapter,C_vs_adapter_aligned_cosine,cifar_10,16,1,True,,0.973853,NaN,diagnostic


In [18]:
# =========================
# 15. Prototype/group-level similarity matrix utilities -- fixed v2
# =========================
# 修正点：
# 1) 不再把 OOD 标签缺失/全为 0 的情况画成 1 x 1 的 OOD class-class。
#    这种图只有一个 prototype 与自己比较，cosine 必然等于 1，没有分析意义。
# 2) ID-OOD 不再压缩成 10 x 1 的 OOD dataset prototype；而是 ID class prototype
#    对 OOD class/group prototypes，得到 10 x K 的矩阵。
# 3) heatmap 默认按本图数值范围自动缩放，并在标题显示 min/mean/max，避免 [-1, 1]
#    色条把 0.85--1.00 的差异全部压成黄色。
# 4) 输出 stats 中记录 ood_group_source/raw_num_ood_labels，便于检查到底用了真实标签还是聚类分组。

PROTOTYPE_ID_PER_CLASS = int(globals().get("PROTOTYPE_ID_PER_CLASS", 20))
PROTOTYPE_OOD_PER_CLASS = int(globals().get("PROTOTYPE_OOD_PER_CLASS", 20))
PROTOTYPE_OOD_GROUPS = int(globals().get("PROTOTYPE_OOD_GROUPS", 10))
PROTOTYPE_OOD_PER_GROUP = int(globals().get("PROTOTYPE_OOD_PER_GROUP", PROTOTYPE_OOD_PER_CLASS))
PROTOTYPE_OOD_MAX_TOTAL_FOR_GROUPING = int(globals().get("PROTOTYPE_OOD_MAX_TOTAL_FOR_GROUPING", 1000))
SIM_HEATMAP_AUTO_SCALE = bool(globals().get("SIM_HEATMAP_AUTO_SCALE", True))
SIM_HEATMAP_PERCENTILES = tuple(globals().get("SIM_HEATMAP_PERCENTILES", (2, 98)))
SIM_HEATMAP_EXCLUDE_DIAGONAL_FOR_SCALE = bool(globals().get("SIM_HEATMAP_EXCLUDE_DIAGONAL_FOR_SCALE", True))

try:
    from sklearn.cluster import MiniBatchKMeans, KMeans
    SKLEARN_CLUSTER_AVAILABLE = True
except Exception as e:
    MiniBatchKMeans = None
    KMeans = None
    SKLEARN_CLUSTER_AVAILABLE = False
    print("[WARN] sklearn.cluster unavailable; OOD pseudo-groups will use deterministic projection bins:", repr(e))


def _sorted_unique_labels(labels):
    labels = np.asarray(labels)
    vals = np.unique(labels)
    try:
        return np.asarray(sorted(vals, key=lambda x: int(x)))
    except Exception:
        return np.asarray(sorted(vals, key=lambda x: str(x)))


def sample_indices_by_label(labels, max_per_label, random_state=2026):
    rng = np.random.default_rng(random_state)
    labels = np.asarray(labels)
    indices = []
    for y in _sorted_unique_labels(labels):
        idx = np.where(labels == y)[0]
        if len(idx) > max_per_label:
            idx = rng.choice(idx, size=max_per_label, replace=False)
        indices.extend(idx.tolist())
    return np.asarray(sorted(indices), dtype=int)


def sample_indices_total(n, max_total, random_state=2026):
    rng = np.random.default_rng(random_state)
    idx = np.arange(int(n))
    if len(idx) > int(max_total):
        idx = rng.choice(idx, size=int(max_total), replace=False)
    return np.asarray(sorted(idx), dtype=int)


def compute_label_prototypes(features, labels):
    """Return L2-normalized group/class prototypes from sample features."""
    x = l2_normalize_np(np.asarray(features, dtype=np.float32))
    y = np.asarray(labels)
    classes = _sorted_unique_labels(y)
    protos = []
    used_classes = []
    for cls in classes:
        m = y == cls
        if not np.any(m):
            continue
        proto = x[m].mean(axis=0, keepdims=True)
        proto = l2_normalize_np(proto)[0]
        protos.append(proto)
        used_classes.append(cls)
    if len(protos) == 0:
        return np.zeros((0, x.shape[1]), dtype=np.float32), np.asarray([])
    return np.stack(protos, axis=0).astype(np.float32), np.asarray(used_classes)


def compute_dataset_prototype(features):
    x = l2_normalize_np(np.asarray(features, dtype=np.float32))
    if len(x) == 0:
        return np.zeros((1, x.shape[1]), dtype=np.float32)
    return l2_normalize_np(x.mean(axis=0, keepdims=True)).astype(np.float32)


def _make_projection_groups(x, n_groups, random_state=2026):
    x = l2_normalize_np(np.asarray(x, dtype=np.float32))
    rng = np.random.default_rng(random_state)
    v = rng.normal(size=(x.shape[1],)).astype(np.float32)
    v = v / max(float(np.linalg.norm(v)), 1e-12)
    scores = x @ v
    order = np.argsort(scores)
    groups = np.zeros(len(x), dtype=int)
    chunks = np.array_split(order, int(n_groups))
    for i, idx in enumerate(chunks):
        groups[idx] = i
    return groups


def make_ood_groups_from_features(x_ood, n_groups=10, random_state=2026):
    """Build deterministic pseudo labels when OOD class labels are unavailable.

    This is not treated as semantic class labels; it only prevents the meaningless
    1 x 1 self-similarity matrix and gives a feature-space group prototype view.
    """
    x = l2_normalize_np(np.asarray(x_ood, dtype=np.float32))
    n = len(x)
    k = int(max(1, min(n_groups, n)))
    if k <= 1:
        return np.zeros(n, dtype=int), "single_group"

    if SKLEARN_CLUSTER_AVAILABLE:
        try:
            x_cluster = x
            if SKLEARN_EMBED_AVAILABLE and PCA is not None and x.shape[1] > 50 and n > 50:
                n_comp = int(min(50, x.shape[1], max(2, n - 1)))
                x_cluster = PCA(n_components=n_comp, random_state=random_state).fit_transform(x)
            if n >= 200:
                km = MiniBatchKMeans(n_clusters=k, random_state=random_state, n_init=10, batch_size=min(512, n))
            else:
                km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
            return km.fit_predict(x_cluster).astype(int), "feature_kmeans"
        except Exception as e:
            print("[WARN] OOD kmeans grouping failed; fallback to projection bins:", repr(e))
    return _make_projection_groups(x, k, random_state=random_state), "projection_bins"


def prepare_similarity_samples(id_payload, ood_payload, branch, random_state=2026):
    x_id = get_feature_np(id_payload, branch)
    x_ood = get_feature_np(ood_payload, branch)
    if x_id is None or x_ood is None:
        return None
    x_id = np.asarray(x_id, dtype=np.float32)
    x_ood = np.asarray(x_ood, dtype=np.float32)

    id_labels = get_labels(id_payload)
    if id_labels is None:
        id_labels = np.zeros(len(x_id), dtype=int)
    id_labels = np.asarray(id_labels).astype(int)

    # Guard against accidental feature/label length mismatch.
    n_id = min(len(x_id), len(id_labels))
    if n_id < len(x_id) or n_id < len(id_labels):
        print(f"[WARN] ID feature/label length mismatch branch={branch}: features={len(x_id)} labels={len(id_labels)}; trimming to {n_id}")
    x_id = x_id[:n_id]
    id_labels = id_labels[:n_id]

    raw_ood_labels = get_labels(ood_payload)
    ood_group_source = "label"
    raw_num_ood_labels = 0
    if raw_ood_labels is not None:
        raw_ood_labels = np.asarray(raw_ood_labels).astype(int)
        n_ood = min(len(x_ood), len(raw_ood_labels))
        if n_ood < len(x_ood) or n_ood < len(raw_ood_labels):
            print(f"[WARN] OOD feature/label length mismatch branch={branch}: features={len(x_ood)} labels={len(raw_ood_labels)}; trimming to {n_ood}")
        x_ood = x_ood[:n_ood]
        raw_ood_labels = raw_ood_labels[:n_ood]
        raw_num_ood_labels = int(len(np.unique(raw_ood_labels)))
    else:
        raw_num_ood_labels = 0

    id_idx = sample_indices_by_label(id_labels, PROTOTYPE_ID_PER_CLASS, random_state=random_state)
    id_idx = id_idx[np.argsort(id_labels[id_idx])]
    x_id_s = np.asarray(x_id[id_idx], dtype=np.float32)
    y_id_s = np.asarray(id_labels[id_idx])

    if raw_ood_labels is not None and len(np.unique(raw_ood_labels)) >= 2:
        ood_idx = sample_indices_by_label(raw_ood_labels, PROTOTYPE_OOD_PER_CLASS, random_state=random_state + 17)
        ood_idx = ood_idx[np.argsort(raw_ood_labels[ood_idx])]
        y_ood_s = np.asarray(raw_ood_labels[ood_idx])
        ood_group_source = "label"
    else:
        max_total = int(min(len(x_ood), PROTOTYPE_OOD_MAX_TOTAL_FOR_GROUPING, PROTOTYPE_OOD_GROUPS * PROTOTYPE_OOD_PER_GROUP * 3))
        base_idx = sample_indices_total(len(x_ood), max_total, random_state=random_state + 17)
        base_x = np.asarray(x_ood[base_idx], dtype=np.float32)
        base_groups, group_source = make_ood_groups_from_features(
            base_x,
            n_groups=PROTOTYPE_OOD_GROUPS,
            random_state=random_state + 101,
        )
        local_idx = sample_indices_by_label(base_groups, PROTOTYPE_OOD_PER_GROUP, random_state=random_state + 23)
        ood_idx = base_idx[local_idx]
        y_ood_s = np.asarray(base_groups[local_idx]).astype(int)
        ood_group_source = group_source if raw_ood_labels is None else f"{group_source}_because_single_label"

    x_ood_s = np.asarray(x_ood[ood_idx], dtype=np.float32)

    id_class_proto, id_classes = compute_label_prototypes(x_id_s, y_id_s)
    ood_group_proto, ood_groups = compute_label_prototypes(x_ood_s, y_ood_s)
    id_dataset_proto = compute_dataset_prototype(x_id_s)
    ood_dataset_proto = compute_dataset_prototype(x_ood_s)

    return {
        "id_samples": l2_normalize_np(x_id_s),
        "ood_samples": l2_normalize_np(x_ood_s),
        "id_sample_labels": y_id_s,
        "ood_sample_groups": y_ood_s,
        "id_class_proto": id_class_proto,
        "ood_group_proto": ood_group_proto,
        "id_dataset_proto": id_dataset_proto,
        "ood_dataset_proto": ood_dataset_proto,
        "id_classes": id_classes,
        "ood_groups": ood_groups,
        "id_idx": id_idx,
        "ood_idx": ood_idx,
        "ood_group_source": ood_group_source,
        "raw_num_ood_labels": int(raw_num_ood_labels),
    }


def cosine_matrix(a, b):
    a = l2_normalize_np(np.asarray(a, dtype=np.float32))
    b = l2_normalize_np(np.asarray(b, dtype=np.float32))
    return np.asarray(a, dtype=np.float64) @ np.asarray(b, dtype=np.float64).T


def _offdiag_values(mat):
    mat = np.asarray(mat)
    if mat.ndim != 2 or mat.shape[0] != mat.shape[1] or mat.shape[0] <= 1:
        return np.asarray([], dtype=float)
    return mat[~np.eye(mat.shape[0], dtype=bool)]


def compute_similarity_pack(id_payload, ood_payload, branch, random_state=2026):
    data = prepare_similarity_samples(id_payload, ood_payload, branch, random_state=random_state)
    if data is None:
        return None

    id_proto = data["id_class_proto"]
    ood_proto = data["ood_group_proto"]
    id_dataset_proto = data["id_dataset_proto"]
    ood_dataset_proto = data["ood_dataset_proto"]

    if id_proto.shape[0] == 0 or ood_proto.shape[0] == 0:
        return None

    s_id_id = cosine_matrix(id_proto, id_proto)
    s_ood_ood = cosine_matrix(ood_proto, ood_proto)
    s_id_ood_group = cosine_matrix(id_proto, ood_proto)
    s_dataset_dataset = cosine_matrix(
        np.vstack([id_dataset_proto, ood_dataset_proto]),
        np.vstack([id_dataset_proto, ood_dataset_proto]),
    )

    yid = data["id_sample_labels"]
    zid = data["id_samples"]
    s_id_samples = cosine_matrix(zid, zid)
    same = yid[:, None] == yid[None, :]
    not_diag = ~np.eye(len(yid), dtype=bool)
    same_no_diag = same & not_diag
    diff = (~same) & not_diag

    id_offdiag = _offdiag_values(s_id_id)
    ood_offdiag = _offdiag_values(s_ood_ood)

    stats = {
        "num_id_classes": int(len(data["id_classes"])),
        "num_ood_groups": int(len(data["ood_groups"])),
        "raw_num_ood_labels": int(data["raw_num_ood_labels"]),
        "ood_group_source": str(data["ood_group_source"]),
        "num_id_samples_used": int(len(data["id_idx"])),
        "num_ood_samples_used": int(len(data["ood_idx"])),
        "samples_per_id_class": int(PROTOTYPE_ID_PER_CLASS),
        "samples_per_ood_group": int(PROTOTYPE_OOD_PER_GROUP),
        # Backward-compatible names.
        "mean_sim_id_id": float(id_offdiag.mean()) if len(id_offdiag) else np.nan,
        "mean_sim_id_same_class": float(s_id_samples[same_no_diag].mean()) if same_no_diag.any() else np.nan,
        "mean_sim_id_diff_class": float(s_id_samples[diff].mean()) if diff.any() else np.nan,
        "mean_sim_ood_ood": float(ood_offdiag.mean()) if len(ood_offdiag) else np.nan,
        "mean_sim_id_ood": float(s_id_ood_group.mean()) if s_id_ood_group.size else np.nan,
        # Explicit names.
        "mean_sim_id_class_proto_offdiag": float(id_offdiag.mean()) if len(id_offdiag) else np.nan,
        "mean_sim_ood_group_proto_offdiag": float(ood_offdiag.mean()) if len(ood_offdiag) else np.nan,
        "mean_sim_id_class_to_ood_group": float(s_id_ood_group.mean()) if s_id_ood_group.size else np.nan,
        "sim_id_dataset_to_ood_dataset": float(s_dataset_dataset[0, 1]),
        "min_sim_id_class_to_ood_group": float(np.nanmin(s_id_ood_group)) if s_id_ood_group.size else np.nan,
        "max_sim_id_class_to_ood_group": float(np.nanmax(s_id_ood_group)) if s_id_ood_group.size else np.nan,
    }
    stats["gap_same_vs_diff"] = stats["mean_sim_id_same_class"] - stats["mean_sim_id_diff_class"]
    stats["gap_idid_vs_idood"] = stats["mean_sim_id_id"] - stats["mean_sim_id_ood"]
    stats["gap_id_class_proto_vs_ood_group"] = (
        stats["mean_sim_id_class_proto_offdiag"] - stats["mean_sim_id_class_to_ood_group"]
    )

    # Backward-compatible aliases used by earlier summary code. They now refer to
    # OOD groups, not necessarily semantic OOD classes. Keeping aliases prevents
    # accidental all-NaN summary columns while the explicit *_group_* columns are
    # the preferred names for interpretation.
    stats["mean_sim_ood_class_proto_offdiag"] = stats["mean_sim_ood_group_proto_offdiag"]
    stats["mean_sim_id_class_to_ood_dataset"] = stats["mean_sim_id_class_to_ood_group"]
    stats["gap_id_class_proto_vs_ood_dataset"] = stats["gap_id_class_proto_vs_ood_group"]

    return {
        "S_ID_ID": s_id_id,
        "S_OOD_OOD": s_ood_ood,
        "S_ID_OOD_GROUP": s_id_ood_group,
        # Alias retained so downstream code does not break; now it is 10 x K groups, not 10 x 1 dataset prototype.
        "S_ID_OOD_DATASET": s_id_ood_group,
        "S_DATASET_DATASET": s_dataset_dataset,
        "id_classes": data["id_classes"],
        "ood_groups": data["ood_groups"],
        "ood_classes": data["ood_groups"],
        "ood_group_source": data["ood_group_source"],
        "stats": stats,
    }


def _format_class_label(x):
    try:
        return str(int(x))
    except Exception:
        return str(x)


def _set_heatmap_ticks(ax, mat, row_labels=None, col_labels=None, max_ticks=30):
    n_rows, n_cols = mat.shape
    if row_labels is not None and n_rows <= max_ticks:
        ax.set_yticks(np.arange(n_rows))
        ax.set_yticklabels([_format_class_label(v) for v in row_labels], fontsize=8)
    else:
        ax.set_yticks([])

    if col_labels is not None and n_cols <= max_ticks:
        ax.set_xticks(np.arange(n_cols))
        ax.set_xticklabels([_format_class_label(v) for v in col_labels], fontsize=8, rotation=90 if n_cols > 10 else 0)
    else:
        ax.set_xticks([])


def _matrix_values_for_scale(mat):
    arr = np.asarray(mat, dtype=float)
    if arr.ndim == 2 and arr.shape[0] == arr.shape[1] and arr.shape[0] > 1 and SIM_HEATMAP_EXCLUDE_DIAGONAL_FOR_SCALE:
        arr = arr[~np.eye(arr.shape[0], dtype=bool)]
    return arr[np.isfinite(arr)].reshape(-1)


def compute_heatmap_limits(packs, keys):
    if not SIM_HEATMAP_AUTO_SCALE:
        return -1.0, 1.0
    vals = []
    for pack in packs:
        if pack is None:
            continue
        for key in keys:
            if key in pack:
                vals.append(_matrix_values_for_scale(pack[key]))
    vals = [v for v in vals if len(v)]
    if not vals:
        return -1.0, 1.0
    arr = np.concatenate(vals)
    lo_p, hi_p = SIM_HEATMAP_PERCENTILES
    lo, hi = np.nanpercentile(arr, [lo_p, hi_p])
    if not np.isfinite(lo) or not np.isfinite(hi):
        return -1.0, 1.0
    if hi - lo < 1e-4:
        center = float(np.nanmean(arr))
        lo, hi = center - 0.01, center + 0.01
    pad = 0.05 * (hi - lo)
    lo = max(-1.0, float(lo - pad))
    hi = min(1.0, float(hi + pad))
    if hi <= lo:
        hi = min(1.0, lo + 0.02)
    return lo, hi


def matrix_stat_title(prefix, mat):
    arr = np.asarray(mat, dtype=float)
    vals = arr[np.isfinite(arr)]
    if len(vals) == 0:
        return prefix
    return f"{prefix}\nmin={np.nanmin(vals):.3f}, mean={np.nanmean(vals):.3f}, max={np.nanmax(vals):.3f}"


def plot_similarity_heatmaps(id_dataset, ood_dataset, shot, seed):
    panels = [
        ("DetBayesRTMMRL", "R", "DetBayesRTMMRL-R"),
        ("DetBayesRTMMRL", "C", "DetBayesRTMMRL-C"),
        ("DetBayesRTMMRL", "MixFeat", "DetBayesRTMMRL-MixFeat"),
        ("BayesAdapter", "adapter", "BayesAdapter"),
    ]
    cols = [
        ("S_ID_ID", "ID class-class"),
        ("S_OOD_OOD", "OOD group-group"),
        ("S_ID_OOD_GROUP", "ID class-OOD group"),
    ]
    nrows, ncols = len(panels), len(cols)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=square_grid_figsize(nrows, ncols, panel_size=FIG_HEATMAP_PANEL_SIZE, extra_width=FIG_COLORBAR_EXTRA_WIDTH, extra_height=FIG_TOP_EXTRA_HEIGHT),
        squeeze=False,
    )
    make_axes_square(axes)
    any_plotted = False
    last_im = None
    stat_rows = []

    pack_by_panel = []
    for method, branch, row_title in panels:
        id_payload, ood_payload = load_case_payloads(method, id_dataset, shot, seed, ood_dataset)
        pack = None
        if id_payload is not None and ood_payload is not None:
            pack = compute_similarity_pack(id_payload, ood_payload, branch, random_state=RANDOM_STATE + int(seed))
        pack_by_panel.append(pack)
        if pack is not None:
            stat_rows.append({
                "method": method,
                "feature_branch": branch,
                "id_dataset": id_dataset,
                "ood_dataset": ood_dataset,
                "shot": int(shot),
                "seed": int(seed),
                **pack["stats"],
            })

    keys = [k for k, _ in cols]
    vmin, vmax = compute_heatmap_limits(pack_by_panel, keys)

    for r, ((method, branch, row_title), pack) in enumerate(zip(panels, pack_by_panel)):
        for c, (key, col_title) in enumerate(cols):
            ax = axes[r, c]
            if pack is None:
                ax.text(0.5, 0.5, "missing feature", ha="center", va="center")
                ax.set_axis_off()
                continue

            mat = pack[key]
            last_im = ax.imshow(mat, aspect="auto", interpolation="nearest", vmin=vmin, vmax=vmax)
            make_axes_square([ax])
            ax.set_title(matrix_stat_title(f"{row_title}: {col_title}", mat), fontsize=10)

            if key == "S_ID_ID":
                _set_heatmap_ticks(ax, mat, row_labels=pack["id_classes"], col_labels=pack["id_classes"], max_ticks=20)
                ax.set_xlabel("ID class")
                ax.set_ylabel("ID class")
            elif key == "S_OOD_OOD":
                _set_heatmap_ticks(ax, mat, row_labels=pack["ood_groups"], col_labels=pack["ood_groups"], max_ticks=25)
                source = pack.get("ood_group_source", "group")
                ax.set_xlabel(f"OOD group ({source})")
                ax.set_ylabel(f"OOD group ({source})")
            else:
                _set_heatmap_ticks(ax, mat, row_labels=pack["id_classes"], col_labels=pack["ood_groups"], max_ticks=25)
                source = pack.get("ood_group_source", "group")
                ax.set_xlabel(f"OOD group ({source})")
                ax.set_ylabel("ID class")
            any_plotted = True

    fig.suptitle(
        f"Prototype/group cosine similarity: {id_dataset} vs {ood_dataset}, shot={shot}, seed={seed}\n"
        f"ID: {PROTOTYPE_ID_PER_CLASS}/class; OOD: labels if available, else {PROTOTYPE_OOD_GROUPS} feature groups; color range=[{vmin:.3f}, {vmax:.3f}]"
    )
    fig.tight_layout(rect=[0, 0.03, 0.91, 0.94])
    cbar_ax = fig.add_axes([0.93, 0.15, 0.018, 0.72])
    if last_im is None:
        last_im = axes[0, 0].imshow([[0]], vmin=vmin, vmax=vmax)
    fig.colorbar(last_im, cax=cbar_ax)

    fig_path = None
    if any_plotted:
        fig_path = FIG_DIRS["similarity_matrix"] / f"prototype_group_sim_matrix_{id_dataset}_{ood_dataset}_shot{shot}_seed{seed}_R_C_MixFeat_adapter_square_v7.png"
        fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return (str(fig_path) if fig_path else None), stat_rows


sim_fig_rows = []
sim_stat_rows = []
for id_dataset in ID_DATASETS:
    for ood_dataset in OOD_DATASETS:
        for shot in SHOTS:
            for seed in SEEDS:
                p, rows = plot_similarity_heatmaps(id_dataset, ood_dataset, shot, seed)
                if p:
                    sim_fig_rows.append({"id_dataset": id_dataset, "ood_dataset": ood_dataset, "shot": shot, "seed": seed, "figure": p})
                sim_stat_rows.extend(rows)

sim_fig_df = pd.DataFrame(sim_fig_rows)
sim_fig_csv = SUMMARY_ROOT / "similarity_matrix_R_C_MixFeat_adapter_figures.csv"
sim_fig_df.to_csv(sim_fig_csv, index=False)

similarity_raw = pd.DataFrame(sim_stat_rows)
sim_raw_csv = SUMMARY_ROOT / "summary_similarity_raw.csv"
similarity_raw.to_csv(sim_raw_csv, index=False)

print("Saved:", sim_fig_csv)
print("Saved:", sim_raw_csv)
if not similarity_raw.empty:
    display(similarity_raw[[c for c in [
        "method", "feature_branch", "id_dataset", "ood_dataset", "shot", "seed",
        "num_id_classes", "num_ood_groups", "raw_num_ood_labels", "ood_group_source",
        "mean_sim_id_id", "mean_sim_ood_ood", "mean_sim_id_ood",
        "min_sim_id_class_to_ood_group", "max_sim_id_class_to_ood_group",
    ] if c in similarity_raw.columns]].head(20))
else:
    display(similarity_raw.head(20))


,method,feature_branch,id_dataset,ood_dataset,shot,seed,num_id_classes,num_ood_groups,raw_num_ood_labels,ood_group_source,mean_sim_id_id,mean_sim_ood_ood,mean_sim_id_ood,min_sim_id_class_to_ood_group,max_sim_id_class_to_ood_group
0,DetBayesRTMMRL,R,cifar_10,dtd,16,1,10,47,47,label,0.850657,0.931389,0.866014,0.743757,0.948833
1,DetBayesRTMMRL,C,cifar_10,dtd,16,1,10,47,47,label,0.874953,0.872812,0.735636,0.629691,0.833629
2,DetBayesRTMMRL,MixFeat,cifar_10,dtd,16,1,10,47,47,label,0.845927,0.890884,0.756621,0.660846,0.856907
3,BayesAdapter,adapter,cifar_10,dtd,16,1,10,47,47,label,0.900297,0.861363,0.743967,0.619692,0.847596


## 15b. 文本特征 / 类别原型 × 各图像特征的跨模态相似度矩阵（原始计算方式）

本节完全保留原 v8 notebook 的计算方式：

- DetBayesRTMMRL-R：R 分支的 ID 类别图像原型和 OOD 图像组原型与 Det 文本类别原型计算余弦相似度；
- DetBayesRTMMRL-C：C 分支采用相同计算；
- DetBayesRTMMRL-MixFeat：沿用原 notebook 的 MixFeat 构造与计算；
- BayesAdapter：CLIP 图像特征原型与 BayesAdapter 后验均值类别原型计算余弦相似度；
- OOD 标签不可用或只有一个标签时，仍沿用原代码的特征分组/KMeans 方式；
- 随机状态、样本数、统计量和自动色条范围均未改变。

唯一变化是：原来的 4×2 大图拆成 8 张独立图，并分别导出 PNG、PDF 和 SVG。PDF/SVG 为矢量格式，SVG 文字保持可编辑。


In [19]:

# =========================
# 15b. Text/class prototype × image feature prototype similarity matrix
# =========================
def get_class_prototypes_np(payload):
    proto = payload.get("class_prototypes", None) if isinstance(payload, dict) else None
    if proto is None:
        return None
    if isinstance(proto, torch.Tensor):
        proto = proto.detach().float().cpu().numpy()
    proto = np.asarray(proto, dtype=np.float32)
    if proto.ndim != 2 or proto.shape[0] == 0:
        return None
    return l2_normalize_np(proto)


def _positive_col_for_id_classes(id_classes, n_text):
    """Map ID class labels to text-prototype columns.

    For CIFAR-style labels, class label k maps to column k. If that is not
    possible, fall back to the row index when the matrix is square.
    """
    cols = []
    for row_i, cls in enumerate(np.asarray(id_classes)):
        col = None
        try:
            c = int(cls)
            if 0 <= c < int(n_text):
                col = c
        except Exception:
            col = None
        if col is None and row_i < int(n_text):
            col = row_i
        cols.append(col)
    return cols


def text_image_matrix_stats(mat, id_classes=None):
    mat = np.asarray(mat, dtype=float)
    vals = mat[np.isfinite(mat)]
    stats = {
        "mean_text_image_sim": float(np.nanmean(vals)) if len(vals) else np.nan,
        "min_text_image_sim": float(np.nanmin(vals)) if len(vals) else np.nan,
        "max_text_image_sim": float(np.nanmax(vals)) if len(vals) else np.nan,
    }

    if id_classes is None or mat.ndim != 2 or mat.shape[0] == 0 or mat.shape[1] == 0:
        stats.update({
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
            "diag_gap": np.nan,
            "margin_mean": np.nan,
            "top1_match_rate": np.nan,
        })
        return stats

    pos_cols = _positive_col_for_id_classes(id_classes, mat.shape[1])
    diag_vals = []
    off_vals = []
    margins = []
    top1_ok = []
    for i, col in enumerate(pos_cols):
        if col is None or i >= mat.shape[0] or col >= mat.shape[1]:
            continue
        row = mat[i]
        finite = np.isfinite(row)
        if not finite[col]:
            continue
        diag_vals.append(float(row[col]))
        neg_mask = np.ones(mat.shape[1], dtype=bool)
        neg_mask[col] = False
        neg = row[neg_mask & np.isfinite(row)]
        if len(neg):
            off_vals.extend(neg.astype(float).tolist())
            margins.append(float(row[col] - np.nanmax(neg)))
        if np.any(finite):
            top1_ok.append(int(np.nanargmax(row) == col))

    diag_vals = np.asarray(diag_vals, dtype=float)
    off_vals = np.asarray(off_vals, dtype=float)
    margins = np.asarray(margins, dtype=float)
    top1_ok = np.asarray(top1_ok, dtype=float)
    stats.update({
        "diag_mean": float(np.nanmean(diag_vals)) if len(diag_vals) else np.nan,
        "offdiag_mean": float(np.nanmean(off_vals)) if len(off_vals) else np.nan,
        "diag_gap": float(np.nanmean(diag_vals) - np.nanmean(off_vals)) if len(diag_vals) and len(off_vals) else np.nan,
        "margin_mean": float(np.nanmean(margins)) if len(margins) else np.nan,
        "top1_match_rate": float(np.nanmean(top1_ok)) if len(top1_ok) else np.nan,
    })
    return stats


def compute_text_image_pack(id_payload, ood_payload, branch, random_state=2026):
    """Build cross-modal matrices: image prototypes × text/class prototypes."""
    text_proto = get_class_prototypes_np(id_payload)
    if text_proto is None:
        text_proto = get_class_prototypes_np(ood_payload)
    if text_proto is None:
        return None

    data = prepare_similarity_samples(id_payload, ood_payload, branch, random_state=random_state)
    if data is None:
        return None

    id_proto = data["id_class_proto"]
    ood_proto = data["ood_group_proto"]
    if id_proto.shape[0] == 0 or ood_proto.shape[0] == 0:
        return None
    if id_proto.shape[1] != text_proto.shape[1]:
        print(
            f"[WARN] text-image dim mismatch branch={branch}: "
            f"image_dim={id_proto.shape[1]} text_dim={text_proto.shape[1]}; skip"
        )
        return None

    s_id_text = cosine_matrix(id_proto, text_proto)
    s_ood_text = cosine_matrix(ood_proto, text_proto)

    id_stats = text_image_matrix_stats(s_id_text, id_classes=data["id_classes"])
    ood_vals = s_ood_text[np.isfinite(s_ood_text)]
    ood_row_max = np.nanmax(s_ood_text, axis=1) if s_ood_text.size else np.asarray([], dtype=float)

    stats = {
        "num_id_classes": int(len(data["id_classes"])),
        "num_text_prototypes": int(text_proto.shape[0]),
        "num_ood_groups": int(len(data["ood_groups"])),
        "ood_group_source": str(data["ood_group_source"]),
        "class_prototype_source": str(id_payload.get("class_prototype_source", "") if isinstance(id_payload, dict) else ""),
        "id_text_diag_mean": id_stats["diag_mean"],
        "id_text_offdiag_mean": id_stats["offdiag_mean"],
        "id_text_diag_gap": id_stats["diag_gap"],
        "id_text_margin_mean": id_stats["margin_mean"],
        "id_text_top1_match_rate": id_stats["top1_match_rate"],
        "id_text_mean": id_stats["mean_text_image_sim"],
        "ood_text_mean": float(np.nanmean(ood_vals)) if len(ood_vals) else np.nan,
        "ood_text_max_mean": float(np.nanmean(ood_row_max)) if len(ood_row_max) else np.nan,
        "ood_text_max_max": float(np.nanmax(ood_row_max)) if len(ood_row_max) else np.nan,
        "id_text_minus_ood_text_max_mean": (
            float(id_stats["diag_mean"] - np.nanmean(ood_row_max))
            if np.isfinite(id_stats.get("diag_mean", np.nan)) and len(ood_row_max)
            else np.nan
        ),
    }

    return {
        "S_ID_TEXT": s_id_text,
        "S_OOD_TEXT": s_ood_text,
        "id_classes": data["id_classes"],
        "ood_groups": data["ood_groups"],
        "ood_group_source": data["ood_group_source"],
        "text_classes": np.arange(text_proto.shape[0]),
        "stats": stats,
    }


def compute_text_image_heatmap_limits(packs):
    vals = []
    for pack in packs:
        if pack is None:
            continue
        for key in ["S_ID_TEXT", "S_OOD_TEXT"]:
            if key in pack:
                arr = np.asarray(pack[key], dtype=float)
                arr = arr[np.isfinite(arr)]
                if len(arr):
                    vals.append(arr)
    if not vals:
        return -1.0, 1.0
    arr = np.concatenate(vals)
    lo_p, hi_p = SIM_HEATMAP_PERCENTILES
    lo, hi = np.nanpercentile(arr, [lo_p, hi_p])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return -1.0, 1.0
    pad = 0.05 * (hi - lo)
    return max(-1.0, float(lo - pad)), min(1.0, float(hi + pad))



def _save_separate_vector_figure(fig, base_path):
    """Save one panel as PNG preview plus editable PDF/SVG vectors."""
    base_path = Path(base_path)
    base_path.parent.mkdir(parents=True, exist_ok=True)

    png_path = base_path.with_suffix(".png")
    pdf_path = base_path.with_suffix(".pdf")
    svg_path = base_path.with_suffix(".svg")

    fig.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")

    return {
        "png": str(png_path),
        "pdf": str(pdf_path),
        "svg": str(svg_path),
    }


def plot_text_image_similarity_heatmaps(id_dataset, ood_dataset, shot, seed):
    """
    Original v8 analysis, with output layout changed only:

    Original computation retained:
      1. DetBayesRTMMRL-R
      2. DetBayesRTMMRL-C
      3. DetBayesRTMMRL-MixFeat
      4. BayesAdapter
      ×
      A. ID image class prototype vs text/class prototype
      B. OOD image group prototype vs text/class prototype

    The former 4x2 composite is exported as eight independent figures.
    All eight figures use the same vmin/vmax calculated by the original
    compute_text_image_heatmap_limits() function.
    """
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    panels = [
        ("DetBayesRTMMRL", "R", "DetBayesRTMMRL-R"),
        ("DetBayesRTMMRL", "C", "DetBayesRTMMRL-C"),
        ("DetBayesRTMMRL", "MixFeat", "DetBayesRTMMRL-MixFeat"),
        ("BayesAdapter", "adapter", "BayesAdapter"),
    ]
    cols = [
        ("S_ID_TEXT", "ID image class × class", "ID"),
        ("S_OOD_TEXT", "OOD image group × class", "OOD"),
    ]

    pack_by_panel = []
    stat_rows = []

    # This block is identical to the original v8 computation.
    for method, branch, row_title in panels:
        id_payload, ood_payload = load_case_payloads(
            method,
            id_dataset,
            shot,
            seed,
            ood_dataset,
        )
        pack = None
        if id_payload is not None and ood_payload is not None:
            pack = compute_text_image_pack(
                id_payload,
                ood_payload,
                branch,
                random_state=RANDOM_STATE + int(seed),
            )
        pack_by_panel.append(pack)

        if pack is not None:
            stat_rows.append({
                "method": method,
                "feature_branch": branch,
                "id_dataset": id_dataset,
                "ood_dataset": ood_dataset,
                "shot": int(shot),
                "seed": int(seed),
                **pack["stats"],
            })

    # Same shared automatic color limits as the original 4x2 figure.
    vmin, vmax = compute_text_image_heatmap_limits(pack_by_panel)

    output_dir = FIG_DIRS["text_image_matrix"] / "separate_vector_original_logic"
    output_dir.mkdir(parents=True, exist_ok=True)

    figure_rows = []

    for (method, branch, row_title), pack in zip(panels, pack_by_panel):
        if pack is None:
            continue

        for key, col_title, split_tag in cols:
            mat = np.asarray(pack[key])

            fig, ax = plt.subplots(
                figsize=(FIG_HEATMAP_PANEL_SIZE + 0.8, FIG_HEATMAP_PANEL_SIZE)
            )

            im = ax.imshow(
                mat,
                aspect="auto",
                interpolation="nearest",
                vmin=vmin,
                vmax=vmax,
            )
            make_axes_square([ax])

            ax.set_title(
                matrix_stat_title(f"{row_title}: {col_title}", mat),
                fontsize=10,
            )

            if key == "S_ID_TEXT":
                _set_heatmap_ticks(
                    ax,
                    mat,
                    row_labels=pack["id_classes"],
                    col_labels=pack["text_classes"],
                    max_ticks=30,
                )
                ax.set_ylabel("ID image class")
            else:
                _set_heatmap_ticks(
                    ax,
                    mat,
                    row_labels=pack["ood_groups"],
                    col_labels=pack["text_classes"],
                    max_ticks=30,
                )
                ax.set_ylabel(
                    f"OOD image group "
                    f"({pack.get('ood_group_source', 'group')})"
                )

            ax.set_xlabel("class prototype")

            # Dedicated colorbar axis: same height as the heatmap.
            divider = make_axes_locatable(ax)
            cax = divider.append_axes("right", size="4.5%", pad=0.12)
            cbar = fig.colorbar(im, cax=cax)
            cbar.set_label("Cosine similarity")

            fig.suptitle(
                f"{id_dataset} vs {ood_dataset}, "
                f"shot={shot}, seed={seed}; "
                ,
                fontsize=10,
                y=0.995,
            )
            fig.tight_layout()

            safe_method = (
                str(method)
                .replace("/", "-")
                .replace(" ", "_")
            )
            safe_branch = (
                str(branch)
                .replace("/", "-")
                .replace(" ", "_")
            )

            base_path = output_dir / (
                f"text_image_sim_"
                f"{safe_method}_{safe_branch}_{split_tag}_"
                f"{id_dataset}_{ood_dataset}_"
                f"shot{shot}_seed{seed}"
            )
            saved = _save_separate_vector_figure(fig, base_path)
            plt.close(fig)

            figure_rows.append({
                "method": method,
                "feature_branch": branch,
                "matrix_key": key,
                "matrix_part": split_tag,
                "id_dataset": id_dataset,
                "ood_dataset": ood_dataset,
                "shot": int(shot),
                "seed": int(seed),
                "vmin": float(vmin),
                "vmax": float(vmax),
                "num_rows": int(mat.shape[0]),
                "num_columns": int(mat.shape[1]),
                **saved,
            })

    return figure_rows, stat_rows


text_image_fig_rows = []
text_image_stat_rows = []

for id_dataset in ID_DATASETS:
    for ood_dataset in OOD_DATASETS:
        for shot in SHOTS:
            for seed in SEEDS:
                fig_rows, stat_rows = plot_text_image_similarity_heatmaps(
                    id_dataset,
                    ood_dataset,
                    shot,
                    seed,
                )
                text_image_fig_rows.extend(fig_rows)
                text_image_stat_rows.extend(stat_rows)

text_image_fig_df = pd.DataFrame(text_image_fig_rows)
text_image_fig_csv = (
    SUMMARY_ROOT
    / "text_image_similarity_matrix_R_C_MixFeat_adapter_"
      "separate_vector_figures.csv"
)
text_image_fig_df.to_csv(text_image_fig_csv, index=False)

text_image_similarity_raw = pd.DataFrame(text_image_stat_rows)
text_image_raw_csv = SUMMARY_ROOT / "summary_text_image_similarity_raw.csv"
text_image_similarity_raw.to_csv(text_image_raw_csv, index=False)

print("Saved:", text_image_fig_csv)
print("Saved:", text_image_raw_csv)

if not text_image_fig_df.empty:
    display(text_image_fig_df)

if not text_image_similarity_raw.empty:
    display(text_image_similarity_raw[[c for c in [
        "method", "feature_branch", "id_dataset", "ood_dataset", "shot", "seed",
        "class_prototype_source", "num_id_classes", "num_text_prototypes",
        "num_ood_groups", "ood_group_source", "id_text_diag_mean",
        "id_text_offdiag_mean", "id_text_diag_gap", "id_text_margin_mean",
        "id_text_top1_match_rate", "ood_text_max_mean",
        "id_text_minus_ood_text_max_mean",
    ] if c in text_image_similarity_raw.columns]])


,method,feature_branch,matrix_key,matrix_part,id_dataset,ood_dataset,shot,seed,vmin,vmax,num_rows,num_columns,png,pdf,svg
0,DetBayesRTMMRL,R,S_ID_TEXT,ID,cifar_10,dtd,16,1,0.031135,0.27041,10,10,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...
1,DetBayesRTMMRL,R,S_OOD_TEXT,OOD,cifar_10,dtd,16,1,0.031135,0.27041,47,10,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...
2,DetBayesRTMMRL,C,S_ID_TEXT,ID,cifar_10,dtd,16,1,0.031135,0.27041,10,10,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...
3,DetBayesRTMMRL,C,S_OOD_TEXT,OOD,cifar_10,dtd,16,1,0.031135,0.27041,47,10,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...
4,DetBayesRTMMRL,MixFeat,S_ID_TEXT,ID,cifar_10,dtd,16,1,0.031135,0.27041,10,10,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...
5,DetBayesRTMMRL,MixFeat,S_OOD_TEXT,OOD,cifar_10,dtd,16,1,0.031135,0.27041,47,10,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...
6,BayesAdapter,adapter,S_ID_TEXT,ID,cifar_10,dtd,16,1,0.031135,0.27041,10,10,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...
7,BayesAdapter,adapter,S_OOD_TEXT,OOD,cifar_10,dtd,16,1,0.031135,0.27041,47,10,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...


,method,feature_branch,id_dataset,ood_dataset,shot,seed,class_prototype_source,num_id_classes,num_text_prototypes,num_ood_groups,ood_group_source,id_text_diag_mean,id_text_offdiag_mean,id_text_diag_gap,id_text_margin_mean,id_text_top1_match_rate,ood_text_max_mean,id_text_minus_ood_text_max_mean
0,DetBayesRTMMRL,R,cifar_10,dtd,16,1,forward_joint.text_features,10,10,47,label,0.151148,0.061431,0.089717,0.054794,1.0,0.091622,0.059527
1,DetBayesRTMMRL,C,cifar_10,dtd,16,1,forward_joint.text_features,10,10,47,label,0.289347,0.193541,0.095806,0.069841,1.0,0.233973,0.055374
2,DetBayesRTMMRL,MixFeat,cifar_10,dtd,16,1,forward_joint.text_features,10,10,47,label,0.314911,0.195249,0.119662,0.083547,1.0,0.237467,0.077444
3,BayesAdapter,adapter,cifar_10,dtd,16,1,model.adapter.get_prototypes(),10,10,47,label,0.286631,0.203570,0.083061,0.055845,1.0,0.238547,0.048085


## 16. Similarity statistics mean/std/delta（含 Det-MixFeat）

`feature_branch` 取值包括：`R`、`C`、`MixFeat`、`adapter`。

这里的 delta 分两类：

1. R vs adapter
2. C vs adapter
3. MixFeat vs adapter

用于分别回答：R/C/MixFeat 分支相对 BayesAdapter 在相似度结构上的差异。

**v3 修复**：`similarity_delta_vs_adapter()` 现在会在 BayesAdapter 或 R/C 分支缺失时返回带标准列的空表，而不是触发 `KeyError: id_dataset`；同时不再强制 BayesAdapter 的 `feature_branch` 必须严格等于 `adapter`。




In [20]:
# =========================
# 16. Similarity statistics summaries
# =========================
SIM_METRIC_COLS = [
    # Backward-compatible sample/prototype names.
    "mean_sim_id_id",
    "mean_sim_id_same_class",
    "mean_sim_id_diff_class",
    "mean_sim_ood_ood",
    "mean_sim_id_ood",
    "gap_same_vs_diff",
    "gap_idid_vs_idood",
    # Explicit prototype/group-level metrics. These are the preferred names.
    "mean_sim_id_class_proto_offdiag",
    "mean_sim_ood_group_proto_offdiag",
    "mean_sim_id_class_to_ood_group",
    "sim_id_dataset_to_ood_dataset",
    "gap_id_class_proto_vs_ood_group",
    "min_sim_id_class_to_ood_group",
    "max_sim_id_class_to_ood_group",
    # Backward-compatible aliases retained for older downstream code.
    "mean_sim_ood_class_proto_offdiag",
    "mean_sim_id_class_to_ood_dataset",
    "gap_id_class_proto_vs_ood_dataset",
]

SIM_MEAN_STD_COLUMNS = ["method", "feature_branch", "id_dataset", "ood_dataset", "shot", "num_seeds"]
for _m in SIM_METRIC_COLS:
    SIM_MEAN_STD_COLUMNS.extend([f"{_m}_mean", f"{_m}_std"])

SIM_DELTA_COLUMNS = [
    "id_dataset",
    "ood_dataset",
    "shot",
    "comparison",
    "metric",
    "DetBayesRTMMRL_branch",
    "DetBayesRTMMRL_value",
    "BayesAdapter_branch",
    "BayesAdapter_value",
    "delta",
]


def empty_similarity_mean_std_df():
    return pd.DataFrame(columns=SIM_MEAN_STD_COLUMNS)


def empty_similarity_delta_df():
    return pd.DataFrame(columns=SIM_DELTA_COLUMNS)


def normalize_branch_name(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def canonical_branch_name(x):
    """Canonicalize feature branch names for exact matching."""
    s = normalize_branch_name(x).replace("-", "").replace("_", "").replace(" ", "").lower()
    if s in {"r", "rep", "imagefeaturesrep", "rbranch"}:
        return "R"
    if s in {"c", "main", "img", "imagefeaturesmain", "cbranch"}:
        return "C"
    if s in {"mixfeat", "mixed", "mixedfeature", "mixedfeatures", "imagemix", "imagefeaturesmix"}:
        return "MixFeat"
    if s in {"adapter", "bayesadapter"}:
        return "adapter"
    return normalize_branch_name(x)


def similarity_mean_std(df):
    """Aggregate similarity statistics over seeds.

    Always returns a DataFrame with the expected columns, even when no usable
    similarity rows exist. This prevents downstream KeyError when some feature
    branches are missing.
    """
    if df is None or df.empty:
        return empty_similarity_mean_std_df()

    required = {"method", "feature_branch", "id_dataset", "ood_dataset", "shot", "seed"}
    missing = sorted(required - set(df.columns))
    if missing:
        print(f"[WARN] similarity_raw is missing required columns: {missing}")
        return empty_similarity_mean_std_df()

    rows = []
    group_cols = ["method", "feature_branch", "id_dataset", "ood_dataset", "shot"]
    for keys, g in df.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys))
        row["num_seeds"] = int(pd.to_numeric(g["seed"], errors="coerce").dropna().nunique())
        for m in SIM_METRIC_COLS:
            vals = pd.to_numeric(g[m], errors="coerce").dropna() if m in g.columns else pd.Series(dtype=float)
            row[f"{m}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{m}_std"] = float(vals.std(ddof=0)) if len(vals) > 1 else (0.0 if len(vals) == 1 else np.nan)
        rows.append(row)

    if not rows:
        return empty_similarity_mean_std_df()
    return pd.DataFrame(rows, columns=SIM_MEAN_STD_COLUMNS).sort_values(
        ["id_dataset", "ood_dataset", "shot", "method", "feature_branch"],
        kind="mergesort",
    ).reset_index(drop=True)


def pick_adapter_row(g):
    """Select the BayesAdapter row for a group.

    Prefer the canonical adapter branch when present, otherwise fall back to the
    first BayesAdapter row and record the branch actually used.
    """
    adapter_rows = g[g["method"].astype(str).eq("BayesAdapter")].copy()
    if adapter_rows.empty:
        return None

    canon = adapter_rows["feature_branch"].map(canonical_branch_name)
    preferred = adapter_rows[canon.eq("adapter")]
    if not preferred.empty:
        return preferred.iloc[0]
    return adapter_rows.iloc[0]


def similarity_delta_vs_adapter(df_mean):
    """Compute DetBayesRTMMRL-R/C/MixFeat minus BayesAdapter deltas.

    The branch matching is canonicalized so that MixFeat is not lost by case
    conversion (e.g. 'MixFeat' -> 'MIXFEAT').
    """
    if df_mean is None or df_mean.empty:
        return empty_similarity_delta_df()

    required = {"method", "feature_branch", "id_dataset", "ood_dataset", "shot"}
    missing = sorted(required - set(df_mean.columns))
    if missing:
        print(f"[WARN] similarity_mean is missing required columns: {missing}")
        return empty_similarity_delta_df()

    rows = []
    id_cols = ["id_dataset", "ood_dataset", "shot"]
    for keys, g in df_mean.groupby(id_cols, dropna=False):
        base = dict(zip(id_cols, keys))
        adapter_row = pick_adapter_row(g)
        if adapter_row is None:
            continue

        canon_branch = g["feature_branch"].map(canonical_branch_name)
        det_mask = g["method"].astype(str).eq("DetBayesRTMMRL")
        for branch in ["R", "C", "MixFeat"]:
            rt = g[det_mask & canon_branch.eq(branch)]
            if rt.empty:
                continue
            rt_row = rt.iloc[0]
            for m in SIM_METRIC_COLS:
                rt_val = pd.to_numeric(rt_row.get(f"{m}_mean", np.nan), errors="coerce")
                ad_val = pd.to_numeric(adapter_row.get(f"{m}_mean", np.nan), errors="coerce")
                delta = np.nan if (pd.isna(rt_val) or pd.isna(ad_val)) else float(rt_val) - float(ad_val)
                rows.append({
                    **base,
                    "comparison": f"DetBayesRTMMRL-{branch}_minus_BayesAdapter",
                    "metric": m,
                    "DetBayesRTMMRL_branch": branch,
                    "DetBayesRTMMRL_value": float(rt_val) if not pd.isna(rt_val) else np.nan,
                    "BayesAdapter_branch": adapter_row.get("feature_branch", "adapter"),
                    "BayesAdapter_value": float(ad_val) if not pd.isna(ad_val) else np.nan,
                    "delta": delta,
                })

    if not rows:
        print(
            "[WARN] No similarity deltas were produced. This usually means "
            "BayesAdapter features were not extracted, or DetBayesRTMMRL R/C/MixFeat branch "
            "features were missing. Check feature_key_report.csv and "
            "summary_similarity_mean_std.csv."
        )
        return empty_similarity_delta_df()

    return pd.DataFrame(rows, columns=SIM_DELTA_COLUMNS).sort_values(
        ["id_dataset", "ood_dataset", "shot", "comparison", "metric"],
        kind="mergesort",
    ).reset_index(drop=True)


similarity_mean = similarity_mean_std(similarity_raw)
sim_mean_csv = SUMMARY_ROOT / "summary_similarity_mean_std.csv"
similarity_mean.to_csv(sim_mean_csv, index=False)

similarity_delta = similarity_delta_vs_adapter(similarity_mean)
sim_delta_csv = SUMMARY_ROOT / "summary_similarity_delta_vs_adapter.csv"
similarity_delta.to_csv(sim_delta_csv, index=False)

print("Saved:", sim_mean_csv)
print("Saved:", sim_delta_csv)
print("similarity_mean shape:", similarity_mean.shape)
print("similarity_delta shape:", similarity_delta.shape)
display(similarity_mean.head(30))
display(similarity_delta.head(30))



,method,feature_branch,id_dataset,ood_dataset,shot,num_seeds,mean_sim_id_id_mean,mean_sim_id_id_std,mean_sim_id_same_class_mean,mean_sim_id_same_class_std,mean_sim_id_diff_class_mean,mean_sim_id_diff_class_std,mean_sim_ood_ood_mean,mean_sim_ood_ood_std,mean_sim_id_ood_mean,mean_sim_id_ood_std,gap_same_vs_diff_mean,gap_same_vs_diff_std,gap_idid_vs_idood_mean,gap_idid_vs_idood_std,mean_sim_id_class_proto_offdiag_mean,mean_sim_id_class_proto_offdiag_std,mean_sim_ood_group_proto_offdiag_mean,mean_sim_ood_group_proto_offdiag_std,mean_sim_id_class_to_ood_group_mean,mean_sim_id_class_to_ood_group_std,sim_id_dataset_to_ood_dataset_mean,sim_id_dataset_to_ood_dataset_std,gap_id_class_proto_vs_ood_group_mean,gap_id_class_proto_vs_ood_group_std,min_sim_id_class_to_ood_group_mean,min_sim_id_class_to_ood_group_std,max_sim_id_class_to_ood_group_mean,max_sim_id_class_to_ood_group_std,mean_sim_ood_class_proto_offdiag_mean,mean_sim_ood_class_proto_offdiag_std,mean_sim_id_class_to_ood_dataset_mean,mean_sim_id_class_to_ood_dataset_std,gap_id_class_proto_vs_ood_dataset_mean,gap_id_class_proto_vs_ood_dataset_std
0,BayesAdapter,adapter,cifar_10,dtd,16,1,0.900297,0.0,0.861823,0.0,0.782004,0.0,0.861363,0.0,0.743967,0.0,0.079819,0.0,0.156330,0.0,0.900297,0.0,0.861363,0.0,0.743967,0.0,0.838542,0.0,0.156330,0.0,0.619692,0.0,0.847596,0.0,0.861363,0.0,0.743967,0.0,0.156330,0.0
1,DetBayesRTMMRL,C,cifar_10,dtd,16,1,0.874953,0.0,0.875488,0.0,0.771393,0.0,0.872812,0.0,0.735636,0.0,0.104094,0.0,0.139317,0.0,0.874953,0.0,0.872812,0.0,0.735636,0.0,0.834398,0.0,0.139317,0.0,0.629691,0.0,0.833629,0.0,0.872812,0.0,0.735636,0.0,0.139317,0.0
2,DetBayesRTMMRL,MixFeat,cifar_10,dtd,16,1,0.845927,0.0,0.888160,0.0,0.755993,0.0,0.890884,0.0,0.756621,0.0,0.132167,0.0,0.089306,0.0,0.845927,0.0,0.890884,0.0,0.756621,0.0,0.862458,0.0,0.089306,0.0,0.660846,0.0,0.856907,0.0,0.890884,0.0,0.756621,0.0,0.089306,0.0
3,DetBayesRTMMRL,R,cifar_10,dtd,16,1,0.850657,0.0,0.926473,0.0,0.791085,0.0,0.931389,0.0,0.866014,0.0,0.135388,0.0,-0.015357,0.0,0.850657,0.0,0.931389,0.0,0.866014,0.0,0.963698,0.0,-0.015357,0.0,0.743757,0.0,0.948833,0.0,0.931389,0.0,0.866014,0.0,-0.015357,0.0


,id_dataset,ood_dataset,shot,comparison,metric,DetBayesRTMMRL_branch,DetBayesRTMMRL_value,BayesAdapter_branch,BayesAdapter_value,delta
0,cifar_10,dtd,16,DetBayesRTMMRL-C_minus_BayesAdapter,gap_id_class_proto_vs_ood_dataset,C,0.139317,adapter,0.156330,-0.017013
1,cifar_10,dtd,16,DetBayesRTMMRL-C_minus_BayesAdapter,gap_id_class_proto_vs_ood_group,C,0.139317,adapter,0.156330,-0.017013
2,cifar_10,dtd,16,DetBayesRTMMRL-C_minus_BayesAdapter,gap_idid_vs_idood,C,0.139317,adapter,0.156330,-0.017013
3,cifar_10,dtd,16,DetBayesRTMMRL-C_minus_BayesAdapter,gap_same_vs_diff,C,0.104094,adapter,0.079819,0.024275
4,cifar_10,dtd,16,DetBayesRTMMRL-C_minus_BayesAdapter,max_sim_id_class_to_ood_group,C,0.833629,adapter,0.847596,-0.013967
5,cifar_10,dtd,16,DetBayesRTMMRL-C_minus_BayesAdapter,mean_sim_id_class_proto_offdiag,C,0.874953,adapter,0.900297,-0.025344
6,cifar_10,dtd,16,DetBayesRTMMRL-C_minus_BayesAdapter,mean_sim_id_class_to_ood_dataset,C,0.735636,adapter,0.743967,-0.008331
7,cifar_10,dtd,16,DetBayesRTMMRL-C_minus_BayesAdapter,mean_sim_id_class_to_ood_group,C,0.735636,adapter,0.743967,-0.008331
8,cifar_10,dtd,16,DetBayesRTMMRL-C_minus_BayesAdapter,mean_sim_id_diff_class,C,0.771393,adapter,0.782004,-0.010611
9,cifar_10,dtd,16,DetBayesRTMMRL-C_minus_BayesAdapter,mean_sim_id_id,C,0.874953,adapter,0.900297,-0.025344


## 17. Similarity distribution 曲线（含 Det-MixFeat）

每张 figure 固定：`ID dataset × OOD dataset × shot`。

布局：

```text
rows = seeds
columns = DetBayesRTMMRL-R / DetBayesRTMMRL-C / DetBayesRTMMRL-MixFeat / BayesAdapter
```

每个 panel 画：

- ID same-class similarity
- ID different-class similarity
- ID-OOD similarity



In [21]:
# =========================
# 17. Prototype/group similarity distribution curves -- fixed v2
# =========================
def similarity_vectors_from_pack(pack):
    id_vals = _offdiag_values(pack["S_ID_ID"])
    ood_vals = _offdiag_values(pack["S_OOD_OOD"])
    id_ood_vals = np.asarray(pack["S_ID_OOD_GROUP"], dtype=float).reshape(-1)
    out = {
        "ID class-class prototypes": id_vals,
        "ID class-OOD group prototypes": id_ood_vals,
    }
    if len(ood_vals):
        out["OOD group-group prototypes"] = ood_vals
    return out


def plot_similarity_distribution_panels(id_dataset, ood_dataset, shot):
    panels = [
        ("DetBayesRTMMRL", "R", "DetBayesRTMMRL-R"),
        ("DetBayesRTMMRL", "C", "DetBayesRTMMRL-C"),
        ("DetBayesRTMMRL", "MixFeat", "DetBayesRTMMRL-MixFeat"),
        ("BayesAdapter", "adapter", "BayesAdapter"),
    ]
    nrows, ncols = len(SEEDS), len(panels)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=square_grid_figsize(nrows, ncols, panel_size=FIG_PANEL_SIZE, extra_height=FIG_TOP_EXTRA_HEIGHT),
        sharex=False,
        sharey=False,
        squeeze=False,
    )
    make_axes_square(axes)
    any_plotted = False

    for r, seed in enumerate(SEEDS):
        for c, (method, branch, title) in enumerate(panels):
            ax = axes[r, c]
            id_payload, ood_payload = load_case_payloads(method, id_dataset, shot, seed, ood_dataset)
            pack = None
            if id_payload is not None and ood_payload is not None:
                pack = compute_similarity_pack(id_payload, ood_payload, branch, random_state=RANDOM_STATE + int(seed))
            if pack is not None:
                vecs = similarity_vectors_from_pack(pack)
                all_vals = []
                for label, vals in vecs.items():
                    vals = np.asarray(vals, dtype=float)
                    vals = vals[np.isfinite(vals)]
                    if len(vals):
                        all_vals.append(vals)
                        plot_distribution_line(ax, vals, label, mode="KDE", value_range=(-1.0, 1.0))
                if all_vals:
                    arr = np.concatenate(all_vals)
                    lo, hi = np.nanpercentile(arr, [1, 99])
                    pad = max(0.01, 0.05 * (hi - lo))
                    ax.set_xlim(max(-1.0, lo - pad), min(1.0, hi + pad))
                any_plotted = True
            ax.set_title(f"{title}, seed={seed}", fontsize=11)
            ax.set_xlabel("Cosine similarity")
            ax.grid(alpha=0.25)
            if r == 0 and c == 0:
                ax.legend(fontsize=9)

    fig.suptitle(
        f"Prototype/group similarity distributions: {id_dataset} vs {ood_dataset}, shot={shot}\n"
        f"ID: {PROTOTYPE_ID_PER_CLASS}/class; OOD: labels if available, else feature groups",
        fontsize=13,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    if not any_plotted:
        plt.close(fig)
        return None
    out_path = FIG_DIRS["similarity_distribution"] / f"prototype_group_sim_distribution_{id_dataset}_{ood_dataset}_shot{shot}_seed_panels_R_C_MixFeat_adapter_square_v7.png"
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return str(out_path)


sim_dist_fig_rows = []
for id_dataset in ID_DATASETS:
    for ood_dataset in OOD_DATASETS:
        for shot in SHOTS:
            p = plot_similarity_distribution_panels(id_dataset, ood_dataset, shot)
            if p:
                sim_dist_fig_rows.append({"id_dataset": id_dataset, "ood_dataset": ood_dataset, "shot": shot, "figure": p})

sim_dist_fig_df = pd.DataFrame(sim_dist_fig_rows)
sim_dist_fig_csv = SUMMARY_ROOT / "similarity_distribution_R_C_MixFeat_adapter_figures.csv"
sim_dist_fig_df.to_csv(sim_dist_fig_csv, index=False)
print("Saved:", sim_dist_fig_csv)
display(sim_dist_fig_df.head(20))


,id_dataset,ood_dataset,shot,figure
0,cifar_10,dtd,16,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 18. ID calibration：reliability diagram 与 ECE/NLL/Brier

Calibration 本身必须基于 confidence，而不是 uncertainty，因为 reliability diagram 比较的是 `confidence` 与 `accuracy`。

这里只对 ID test set 做 calibration。



In [22]:

# =========================
# 18. Calibration metrics and reliability diagrams
# =========================
def calibration_metrics_from_payload(payload, n_bins=15):
    labels = payload.get("labels", None)
    logits = payload.get("logits", None)
    conf = payload.get("msp_confidence", None)
    preds = payload.get("preds", None)
    if labels is None or logits is None or conf is None or preds is None:
        return None
    y = to_numpy_1d(labels).astype(int)
    c = to_numpy_1d(conf).astype(float)
    pred = to_numpy_1d(preds).astype(int)
    correct = (pred == y).astype(float)
    probs = probs_from_logits_tensor(logits).detach().cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    mce = 0.0
    bin_rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (c >= lo) & (c < hi if i < n_bins - 1 else c <= hi)
        if not mask.any():
            bin_rows.append({"bin": i, "lo": lo, "hi": hi, "count": 0, "confidence": np.nan, "accuracy": np.nan})
            continue
        acc = float(correct[mask].mean())
        avg_conf = float(c[mask].mean())
        gap = abs(acc - avg_conf)
        ece += float(mask.mean()) * gap
        mce = max(mce, gap)
        bin_rows.append({"bin": i, "lo": lo, "hi": hi, "count": int(mask.sum()), "confidence": avg_conf, "accuracy": acc})

    try:
        nll = float(log_loss(y, probs, labels=list(range(probs.shape[1]))))
    except Exception:
        nll = np.nan
    try:
        # Multi-class Brier: mean sum_c (p_c - onehot_c)^2
        onehot = np.eye(probs.shape[1])[y]
        brier = float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))
    except Exception:
        brier = np.nan

    return {
        "ECE": float(ece),
        "MCE": float(mce),
        "NLL": nll,
        "Brier": brier,
        "Accuracy": float(correct.mean()),
        "AvgConfidence": float(c.mean()),
        "bin_rows": bin_rows,
    }

calib_rows = []
calib_bin_rows = []
for id_dataset in ID_DATASETS:
    for shot in SHOTS:
        for seed in SEEDS:
            for method in ["DetBayesRTMMRL", "BayesAdapter"]:
                cache_dir = build_case_cache_dir(method, id_dataset, shot, seed)
                id_path = cache_dir / "id_test_outputs.pt"
                if not id_path.exists():
                    continue
                payload = load_tensor_payload(id_path)
                m = calibration_metrics_from_payload(payload, n_bins=15)
                if m is None:
                    continue
                row = {"method": method, "id_dataset": id_dataset, "shot": int(shot), "seed": int(seed)}
                for k in ["ECE", "MCE", "NLL", "Brier", "Accuracy", "AvgConfidence"]:
                    row[k] = m[k]
                calib_rows.append(row)
                for br in m["bin_rows"]:
                    calib_bin_rows.append({**row, **br})

calibration_raw = pd.DataFrame(calib_rows)
calib_raw_csv = SUMMARY_ROOT / "summary_calibration_raw.csv"
calibration_raw.to_csv(calib_raw_csv, index=False)

calibration_bins = pd.DataFrame(calib_bin_rows)
calib_bins_csv = SUMMARY_ROOT / "summary_calibration_bins.csv"
calibration_bins.to_csv(calib_bins_csv, index=False)

calibration_mean = mean_std_summary(
    calibration_raw.assign(ood_dataset="", score_name="msp_confidence", status="ok"),
    ["ECE", "MCE", "NLL", "Brier", "Accuracy", "AvgConfidence"]
) if not calibration_raw.empty else pd.DataFrame()
calib_mean_csv = SUMMARY_ROOT / "summary_calibration_mean_std.csv"
calibration_mean.to_csv(calib_mean_csv, index=False)

print("Saved:", calib_raw_csv)
print("Saved:", calib_bins_csv)
print("Saved:", calib_mean_csv)
display(calibration_raw.head(20))



,method,id_dataset,shot,seed,ECE,MCE,NLL,Brier,Accuracy,AvgConfidence
0,DetBayesRTMMRL,cifar_10,16,1,0.011386,0.320993,0.170268,0.082640,0.9458,0.95489
1,BayesAdapter,cifar_10,16,1,0.017912,0.101143,0.204399,0.097227,0.9334,0.91639


In [23]:
# =========================
# 18b. Reliability diagram seed panels
# =========================
def plot_reliability_seed_panels(id_dataset, shot):
    if calibration_bins.empty:
        return None
    nrows, ncols = 1, len(SEEDS)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=square_grid_figsize(nrows, ncols, panel_size=FIG_PANEL_SIZE, extra_height=FIG_TOP_EXTRA_HEIGHT),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    make_axes_square(axes)
    axes_flat = axes.reshape(-1)
    any_plotted = False
    for ax, seed in zip(axes_flat, SEEDS):
        for method, linestyle in [("DetBayesRTMMRL", "-"), ("BayesAdapter", "--")]:
            g = calibration_bins[(calibration_bins["id_dataset"] == id_dataset) & (calibration_bins["shot"] == int(shot)) & (calibration_bins["seed"] == int(seed)) & (calibration_bins["method"] == method)]
            g = g.dropna(subset=["confidence", "accuracy"])
            if g.empty:
                continue
            ax.plot(g["confidence"], g["accuracy"], marker="o", linestyle=linestyle, label=method)
            any_plotted = True
        ax.plot([0, 1], [0, 1], linestyle=":", linewidth=1.0, label="perfect" if seed == SEEDS[0] else None)
        ax.set_title(f"seed={seed}", fontsize=11)
        ax.set_xlabel("MSP confidence")
        ax.set_ylabel("Accuracy")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.25)
    axes_flat[0].legend(fontsize=9)
    fig.suptitle(f"Reliability diagram: {id_dataset}, shot={shot}", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    if not any_plotted:
        plt.close(fig)
        return None
    out_path = FIG_DIRS["calibration"] / f"reliability_{id_dataset}_shot{shot}_seed_panels_square_v7.png"
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return str(out_path)

rel_fig_rows = []
for id_dataset in ID_DATASETS:
    for shot in SHOTS:
        p = plot_reliability_seed_panels(id_dataset, shot)
        if p:
            rel_fig_rows.append({"id_dataset": id_dataset, "shot": shot, "figure": p})
rel_fig_df = pd.DataFrame(rel_fig_rows)
rel_fig_csv = SUMMARY_ROOT / "reliability_figures.csv"
rel_fig_df.to_csv(rel_fig_csv, index=False)
print("Saved:", rel_fig_csv)
display(rel_fig_df.head(20))


,id_dataset,shot,figure
0,cifar_10,16,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 19. Risk-Coverage Curve / AURC

基于 ID test set：按 MSP uncertainty 从低到高排序，保留低不确定样本，计算 coverage 与 risk。



In [24]:

# =========================
# 19. Risk-coverage
# =========================
def risk_coverage_curve(correct_bool, uncertainty):
    correct_bool = np.asarray(correct_bool).astype(bool)
    uncertainty = np.asarray(uncertainty, dtype=float)
    order = np.argsort(uncertainty)  # low uncertainty retained first
    corr_sorted = correct_bool[order]
    n = len(corr_sorted)
    if n == 0:
        return np.array([]), np.array([])
    coverages = np.arange(1, n + 1) / n
    errors = (~corr_sorted).astype(float)
    risks = np.cumsum(errors) / np.arange(1, n + 1)
    return coverages, risks


def aurc_eaurc(correct_bool, uncertainty):
    cov, risk = risk_coverage_curve(correct_bool, uncertainty)
    if len(cov) == 0:
        return np.nan, np.nan
    aurc = float(np.trapz(risk, cov))
    # Simplified excess AURC: subtract oracle risk-coverage integral.
    errors = (~np.asarray(correct_bool).astype(bool)).astype(float)
    oracle_order = np.argsort(errors)  # correct first, errors last
    oracle_corr = np.asarray(correct_bool).astype(bool)[oracle_order]
    oracle_cov = np.arange(1, len(oracle_corr) + 1) / len(oracle_corr)
    oracle_risk = np.cumsum((~oracle_corr).astype(float)) / np.arange(1, len(oracle_corr) + 1)
    oracle_aurc = float(np.trapz(oracle_risk, oracle_cov))
    return aurc, aurc - oracle_aurc

risk_rows = []
for id_dataset in ID_DATASETS:
    for shot in SHOTS:
        for seed in SEEDS:
            for method in ["DetBayesRTMMRL", "BayesAdapter"]:
                cache_dir = build_case_cache_dir(method, id_dataset, shot, seed)
                id_path = cache_dir / "id_test_outputs.pt"
                if not id_path.exists():
                    continue
                payload = load_tensor_payload(id_path)
                correct = get_correct(payload)
                unc = get_unc(payload)
                if correct is None or len(correct) != len(unc):
                    continue
                aurc, eaurc = aurc_eaurc(correct, unc)
                risk_rows.append({
                    "method": method,
                    "id_dataset": id_dataset,
                    "shot": int(shot),
                    "seed": int(seed),
                    "score_name": "msp_uncertainty",
                    "AURC": aurc,
                    "EAURC": eaurc,
                })

risk_raw = pd.DataFrame(risk_rows)
risk_raw_csv = SUMMARY_ROOT / "summary_risk_coverage_msp_raw.csv"
risk_raw.to_csv(risk_raw_csv, index=False)

risk_mean = mean_std_summary(
    risk_raw.assign(ood_dataset="", status="ok"),
    ["AURC", "EAURC"]
) if not risk_raw.empty else pd.DataFrame()
risk_mean_csv = SUMMARY_ROOT / "summary_risk_coverage_msp_mean_std.csv"
risk_mean.to_csv(risk_mean_csv, index=False)

print("Saved:", risk_raw_csv)
print("Saved:", risk_mean_csv)
display(risk_raw.head(20))



/tmp/ipykernel_10693/4265753473.py:22: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  aurc = float(np.trapz(risk, cov))
/tmp/ipykernel_10693/4265753473.py:29: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  oracle_aurc = float(np.trapz(oracle_risk, oracle_cov))


,method,id_dataset,shot,seed,score_name,AURC,EAURC
0,DetBayesRTMMRL,cifar_10,16,1,msp_uncertainty,0.006326,0.004830
1,BayesAdapter,cifar_10,16,1,msp_uncertainty,0.008804,0.006535


In [25]:
# =========================
# 19b. Risk-coverage figures
# =========================
def plot_risk_coverage_seed_panels(id_dataset, shot):
    nrows, ncols = 1, len(SEEDS)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=square_grid_figsize(nrows, ncols, panel_size=FIG_PANEL_SIZE, extra_height=FIG_TOP_EXTRA_HEIGHT),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    make_axes_square(axes)
    axes_flat = axes.reshape(-1)
    any_plotted = False
    for ax, seed in zip(axes_flat, SEEDS):
        for method, linestyle in [("DetBayesRTMMRL", "-"), ("BayesAdapter", "--")]:
            cache_dir = build_case_cache_dir(method, id_dataset, shot, seed)
            id_path = cache_dir / "id_test_outputs.pt"
            if not id_path.exists():
                continue
            payload = load_tensor_payload(id_path)
            correct = get_correct(payload)
            unc = get_unc(payload)
            if correct is None or len(correct) != len(unc):
                continue
            cov, risk = risk_coverage_curve(correct, unc)
            ax.plot(cov, risk, linestyle=linestyle, label=method)
            any_plotted = True
        ax.set_title(f"seed={seed}", fontsize=11)
        ax.set_xlabel("Coverage")
        ax.set_ylabel("Risk")
        ax.set_xlim(0, 1)
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.25)
    axes_flat[0].legend(fontsize=9)
    fig.suptitle(f"Risk-Coverage Curve: {id_dataset}, shot={shot}, score=MSP uncertainty", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    if not any_plotted:
        plt.close(fig)
        return None
    out_path = FIG_DIRS["risk_coverage"] / f"risk_coverage_{id_dataset}_shot{shot}_seed_panels_square_v7.png"
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return str(out_path)

risk_fig_rows = []
for id_dataset in ID_DATASETS:
    for shot in SHOTS:
        p = plot_risk_coverage_seed_panels(id_dataset, shot)
        if p:
            risk_fig_rows.append({"id_dataset": id_dataset, "shot": shot, "figure": p})
risk_fig_df = pd.DataFrame(risk_fig_rows)
risk_fig_csv = SUMMARY_ROOT / "risk_coverage_figures.csv"
risk_fig_df.to_csv(risk_fig_csv, index=False)
print("Saved:", risk_fig_csv)
display(risk_fig_df.head(20))


,id_dataset,shot,figure
0,cifar_10,16,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 20. MSP OOD metric delta 图

这里画基于 MSP uncertainty 的 OOD 指标 delta。

`delta = DetBayesRTMMRL - BayesAdapter`，但不同指标方向不同，因此图表索引里保留 `metric_direction`。



In [26]:

# =========================
# 20. Delta metric figures
# =========================
def plot_delta_metric(delta_df: pd.DataFrame, metric: str):
    if delta_df is None or delta_df.empty:
        return None
    required = {"status", "metric", "id_dataset", "ood_dataset", "shot", "delta_DetBayesRTMMRL_minus_BayesAdapter"}
    if not required.issubset(set(delta_df.columns)):
        print(f"[WARN] delta_df missing columns for plot_delta_metric: {sorted(required - set(delta_df.columns))}")
        return None
    ok = delta_df[(delta_df["status"] == "ok") & (delta_df["metric"] == metric)].copy()
    if ok.empty:
        return None
    ok["case"] = ok["id_dataset"].astype(str) + "→" + ok["ood_dataset"].astype(str) + " s" + ok["shot"].astype(str)
    ok = ok.sort_values(["id_dataset", "ood_dataset", "shot"])
    col = "delta_DetBayesRTMMRL_minus_BayesAdapter"
    fig_path = FIG_DIRS["metric_delta"] / f"delta_msp_{metric}_large_v7.png"
    plt.figure(figsize=(max(10, len(ok) * 0.55), 6.2))
    plt.bar(np.arange(len(ok)), ok[col].astype(float).to_numpy())
    plt.axhline(0.0, linewidth=1)
    plt.xticks(np.arange(len(ok)), ok["case"].tolist(), rotation=70, ha="right")
    plt.ylabel(f"DetBayesRTMMRL - BayesAdapter ({metric})")
    plt.title(f"OOD delta based on MSP uncertainty: {metric} ({metric_direction(metric)})")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()
    return str(fig_path)

delta_fig_rows = []
for metric in ["AUROC", "AUPR_OUT", "AUPR_IN", "FPR95", "DetectionError"]:
    p = plot_delta_metric(ood_delta, metric)
    if p:
        delta_fig_rows.append({"metric": metric, "metric_direction": metric_direction(metric), "figure": p})

delta_fig_df = pd.DataFrame(delta_fig_rows)
delta_fig_csv = SUMMARY_ROOT / "delta_msp_figures.csv"
delta_fig_df.to_csv(delta_fig_csv, index=False)
print("Saved:", delta_fig_csv)
display(delta_fig_df)


,metric,metric_direction,figure
0,AUROC,higher_better,/root/autodl-tmp/MMRL/output_refactor/analysis...
1,AUPR_OUT,higher_better,/root/autodl-tmp/MMRL/output_refactor/analysis...
2,AUPR_IN,higher_better,/root/autodl-tmp/MMRL/output_refactor/analysis...
3,FPR95,lower_better,/root/autodl-tmp/MMRL/output_refactor/analysis...
4,DetectionError,lower_better,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 21. 输出索引

最终索引会记录所有 summary、figure index 和 cache root。



In [27]:

# =========================
# 21. 输出索引
# =========================
outputs = {
    "notebook_version": NOTEBOOK_VERSION,
    "analysis_root": str(ANALYSIS_ROOT),
    "cache_root": str(CACHE_ROOT),
    "prediction_cache_root": str(PRED_CACHE_ROOT),
    "feature_cache_root": str(FEATURE_CACHE_ROOT),
    "summary_root": str(SUMMARY_ROOT),
    "figure_root": str(FIGURE_ROOT),
    "paper_ready_root": str(PAPER_READY_ROOT),
    "main_uncertainty_score": MAIN_UNCERTAINTY_SCORE,
    "summary_ood_msp_raw": str(SUMMARY_ROOT / "summary_ood_msp_raw.csv"),
    "summary_ood_msp_mean_std": str(SUMMARY_ROOT / "summary_ood_msp_mean_std.csv"),
    "summary_ood_msp_delta": str(SUMMARY_ROOT / "summary_ood_msp_delta.csv"),
    "summary_similarity_raw": str(SUMMARY_ROOT / "summary_similarity_raw.csv"),
    "summary_text_image_similarity_raw": str(SUMMARY_ROOT / "summary_text_image_similarity_raw.csv"),
    "summary_similarity_mean_std": str(SUMMARY_ROOT / "summary_similarity_mean_std.csv"),
    "summary_similarity_delta_vs_adapter": str(SUMMARY_ROOT / "summary_similarity_delta_vs_adapter.csv"),
    "summary_calibration_raw": str(SUMMARY_ROOT / "summary_calibration_raw.csv"),
    "summary_calibration_mean_std": str(SUMMARY_ROOT / "summary_calibration_mean_std.csv"),
    "summary_risk_coverage_msp_raw": str(SUMMARY_ROOT / "summary_risk_coverage_msp_raw.csv"),
    "summary_risk_coverage_msp_mean_std": str(SUMMARY_ROOT / "summary_risk_coverage_msp_mean_std.csv"),
    "figure_indices": {
        "msp_uncertainty_distribution": str(SUMMARY_ROOT / "msp_uncertainty_distribution_figures.csv"),
        "msp_correct_wrong_ood": str(SUMMARY_ROOT / "msp_correct_wrong_ood_figures.csv"),
        "feature_embedding_R_C_MixFeat_adapter": str(SUMMARY_ROOT / "feature_embedding_R_C_MixFeat_adapter_figures.csv"),
        "similarity_matrix_R_C_MixFeat_adapter": str(SUMMARY_ROOT / "similarity_matrix_R_C_MixFeat_adapter_figures.csv"),
        "text_image_similarity_matrix_R_C_MixFeat_adapter": str(SUMMARY_ROOT / "text_image_similarity_matrix_R_C_MixFeat_adapter_separate_vector_figures.csv"),
        "similarity_distribution_R_C_MixFeat_adapter": str(SUMMARY_ROOT / "similarity_distribution_R_C_MixFeat_adapter_figures.csv"),
        "reliability": str(SUMMARY_ROOT / "reliability_figures.csv"),
        "risk_coverage": str(SUMMARY_ROOT / "risk_coverage_figures.csv"),
        "delta_msp": str(SUMMARY_ROOT / "delta_msp_figures.csv"),
    },
    "figure_dirs": {k: str(v) for k, v in FIG_DIRS.items()},
}

index_path = SUMMARY_ROOT / "output_index.json"
with index_path.open("w", encoding="utf-8") as f:
    json.dump(outputs, f, indent=2, ensure_ascii=False)

print(json.dumps(outputs, indent=2, ensure_ascii=False))

